# Publication copy

Notebook outputs were cleared for distribution. Scientific settings are preserved; see `../docs/REPRODUCIBILITY.md` before interpreting numerical intervals. Run with the notebook working directory set to `plot/`.


# Full-data preview with ten additional faint-end measurements

# Display revision: With PBH and No PBH only

# PRL seven-panel galaxy UV luminosity functions at z=6-12.5 — clean limit-errorbar alternative

## Objective and evidence policy

Build a reproducible, auditable seven-panel UVLF comparison for a PRL two-column figure. This alternative uses 1.0 mm central markers, transparent interiors for comparison markers, 1.5 pt error-bar caps, and unfilled 0.5 pt line arrows drawn directly on the data/error-bar x coordinate for in-range one-sided limits. Every reported uncertainty remains unchanged in the source ledger; for a one-sided limit, only the vertical error-bar arm in the arrow direction is suppressed at rendering time to avoid overplotting, while any reported opposite-direction arm is retained. The Curtis-Lake Deep sample uses a neutral open hexagon to avoid conflating dataset identity with limit direction. One Harikane lower-limit row whose central numerical anchor is below the fixed y-axis range is explicitly excluded from display; the four Chemerynska rows centered at the right boundary remain plotted with only their out-of-frame positive x-error segments clipped. No boundary-continuation arrows are drawn. The original variants remain separate. All numerical inputs are read from `UVLF_literature_data.txt`; this notebook does not hand-enter data values.

## Dependencies

The notebook uses pathlib, NumPy, pandas, and Matplotlib. Figure-specific typography and strokes are scoped to the figure constructor so global interactive defaults remain unchanged.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path("UVLF_literature_data.txt")
OUTPUT_STEM = Path("UVLF_full_WithPBH_noRobertson_20261016")
PANELS = (6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.5)
X_LIMITS = (-24.0, -15.5)
Y_LIMITS = (1e-8, 5e-2)
MIN_VISUAL_EXTENT_MM = 2.0
MIN_VISUAL_EXTENT_PT = 72 * MIN_VISUAL_EXTENT_MM / 25.4
SEMANTIC_MARKER_EXTENT_PT = 1.02 * MIN_VISUAL_EXTENT_PT
CENTRAL_MARKER_EXTENT_MM = 1.0
CENTRAL_MARKER_EXTENT_PT = 72 * CENTRAL_MARKER_EXTENT_MM / 25.4
CENTRAL_MARKER_EQUALITY_TOLERANCE_PT = 1e-9
LIMIT_ARROW_OFFSET_PT = 7.0
LIMIT_ARROW_LINEWIDTH_PT = 0.5
LIMIT_ARROW_MUTATION_SCALE_PT = 5.0
LIMIT_ARROW_X_OFFSET_PT = 0.0
BOUNDARY_ARROW_MUTATION_SCALE_PT = 14.6
ARROW_MARKER_CLEARANCE_MM = 0.25
RIGHT_ARROW_OFFSET_CANDIDATES_PT = ((0.0, -8.0), (0.0, 8.0), (0.0, -7.0), (0.0, 7.0), (0.0, -9.0), (0.0, 9.0), (0.0, -10.0), (0.0, 10.0), (0.0, -12.0), (0.0, 12.0))
LOWER_ARROW_OFFSET_CANDIDATES_PT = ((7.0, 2.0), (8.0, 2.0), (9.0, 3.0), (12.0, 6.0))
ARTIFACT_TIMESTAMP = datetime(2026, 8, 26, tzinfo=timezone.utc)
REQUIRED_COLUMNS = (
    "dataset_id", "reference", "year", "doi_or_arxiv",
    "publication_status", "source_table", "redshift_panel",
    "z_effective", "z_min", "z_max", "M_UV",
    "M_UV_err_minus", "M_UV_err_plus", "phi",
    "phi_err_minus", "phi_err_plus", "limit_type",
    "uncertainty_definition", "plot_flag", "notes",
)

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "svg.hashsalt": "UVLF_full_WithPBH_noRobertson_20261016",
})


In [ ]:
# Loader and validator
def load_uvlf_data(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, sep="\t", comment="#")
    for column in REQUIRED_COLUMNS:
        if column not in frame.columns:
            raise ValueError(f"Missing required column: {column}")
    frame["plot_flag"] = frame["plot_flag"].astype(str).str.lower().map({"true": True, "false": False})
    if frame["plot_flag"].isna().any():
        raise ValueError("plot_flag must contain only true or false")
    return frame


def validate_uvlf_data(frame: pd.DataFrame) -> None:
    missing = set(REQUIRED_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    if not pd.api.types.is_bool_dtype(frame["plot_flag"]) or frame["plot_flag"].isna().any():
        raise ValueError("plot_flag must contain only boolean values")
    plotted = frame.loc[frame["plot_flag"]].copy()
    required_values = ["dataset_id", "reference", "source_table", "redshift_panel", "M_UV", "phi", "limit_type"]
    if plotted[required_values].isna().any().any():
        raise ValueError("Plotted rows contain missing required values")
    if (plotted["phi"] <= 0).any():
        raise ValueError("phi must be positive for plotted rows")
    error_columns = ["M_UV_err_minus", "M_UV_err_plus", "phi_err_minus", "phi_err_plus"]
    if (plotted[error_columns].fillna(0) < 0).any().any():
        raise ValueError("Error magnitudes must be non-negative")
    if not set(plotted["limit_type"]).issubset({"detection", "upper", "lower"}):
        raise ValueError("Unrecognized limit_type")
    if not set(plotted["redshift_panel"].astype(float)).issubset(PANELS):
        raise ValueError("Invalid redshift_panel")
    duplicates = plotted.duplicated(["dataset_id", "redshift_panel", "M_UV", "phi"])
    if duplicates.any():
        raise ValueError("Duplicate plotted source rows")


def summarize_uvlf_data(frame: pd.DataFrame) -> pd.DataFrame:
    plotted = frame.loc[frame["plot_flag"]]
    return (
        plotted.groupby(["redshift_panel", "dataset_id"], observed=True)
        .size().rename("n_points").reset_index()
        .sort_values(["redshift_panel", "dataset_id"])
    )


In [ ]:
# Validator self-tests: one valid case and five required negative cases.
def minimal_valid_row():
    return {
        "dataset_id": "test_dataset", "reference": "Test et al.",
        "year": 2026, "doi_or_arxiv": "test",
        "publication_status": "test", "source_table": "Table 1",
        "redshift_panel": 6.0, "z_effective": 6.0, "z_min": np.nan,
        "z_max": np.nan, "M_UV": -20.0, "M_UV_err_minus": np.nan,
        "M_UV_err_plus": np.nan, "phi": 1e-4,
        "phi_err_minus": 1e-5, "phi_err_plus": 1e-5,
        "limit_type": "detection", "uncertainty_definition": "test",
        "plot_flag": True, "notes": "",
    }


def minimal_valid_frame():
    return pd.DataFrame([minimal_valid_row()], columns=REQUIRED_COLUMNS)


def expect_validation_error(frame, message_fragment):
    try:
        validate_uvlf_data(frame)
    except ValueError as exc:
        assert message_fragment in str(exc)
    else:
        raise AssertionError(f"Expected ValueError containing {message_fragment!r}")


def test_valid_row_passes():
    validate_uvlf_data(minimal_valid_frame())


def test_malformed_plot_flag_fails():
    frame = minimal_valid_frame().astype({"plot_flag": object})
    frame.loc[0, "plot_flag"] = "yes"
    try:
        validate_uvlf_data(frame)
    except ValueError as exc:
        assert "plot_flag must contain only boolean values" in str(exc)
    except KeyError as exc:
        raise AssertionError("validate_uvlf_data must reject malformed plot_flag before row selection") from exc
    else:
        raise AssertionError("Expected ValueError for malformed plot_flag")


def test_missing_phi_fails():
    frame = minimal_valid_frame()
    frame.loc[0, "phi"] = np.nan
    expect_validation_error(frame, "Plotted rows contain missing required values")


def test_nonpositive_phi_fails():
    frame = minimal_valid_frame()
    frame.loc[0, "phi"] = 0.0
    expect_validation_error(frame, "phi must be positive")


def test_negative_error_fails():
    frame = minimal_valid_frame()
    frame.loc[0, "phi_err_minus"] = -1e-5
    expect_validation_error(frame, "Error magnitudes must be non-negative")


def test_unknown_limit_type_fails():
    frame = minimal_valid_frame()
    frame.loc[0, "limit_type"] = "unknown"
    expect_validation_error(frame, "Unrecognized limit_type")


def test_invalid_panel_fails():
    frame = minimal_valid_frame()
    frame.loc[0, "redshift_panel"] = 6.5
    expect_validation_error(frame, "Invalid redshift_panel")


test_valid_row_passes()
test_malformed_plot_flag_fails()
test_missing_phi_fails()
test_nonpositive_phi_fails()
test_negative_error_fails()
test_unknown_limit_type_fails()
test_invalid_panel_fails()
print("Validator self-tests passed: 1 valid case and 6 negative cases.")


In [ ]:
# Load and validate the audited ledger.
data = load_uvlf_data(DATA_PATH)
validate_uvlf_data(data)
display_exclusion_mask = (
    data["plot_flag"]
    & data["dataset_id"].eq("harikane2025_spec")
    & data["redshift_panel"].astype(float).eq(7.0)
    & data["M_UV"].astype(float).eq(-23.7)
    & data["phi"].astype(float).lt(Y_LIMITS[0])
)
display_excluded = data.loc[display_exclusion_mask].copy()
assert len(display_excluded) == 1
figure_data = data.copy()
figure_data.loc[display_exclusion_mask, "plot_flag"] = False
validate_uvlf_data(figure_data)


In [ ]:
summary = summarize_uvlf_data(figure_data)
display(summary)

primary = figure_data.loc[figure_data["dataset_id"].isin({"bouwens2021_primary", "donnan2024_primary"}) & figure_data["plot_flag"]]
assert len(primary) == 55
assert primary.groupby("redshift_panel").size().to_dict() == {
    6.0: 10, 7.0: 10, 8.0: 7, 9.0: 7,
    10.0: 7, 11.0: 7, 12.5: 7,
}
print(f"Validated {len(data)} ledger rows; {len(primary)} primary plotted rows across {len(PANELS)} panels; display-excluded={len(display_excluded)}.")


## Updated PBH merger models
The solid and dashed curves use the supplied 8- and 16-parameter models with PBHs merging at z=20. The solid model uses the fixed astrophysics in the new script. Both use total-matter fluctuations. The dotted no-PBH reference retains its model and uses the newly supplied no-PBH astrophysical parameters. All curves are returned to the published observational frame using the inverse dust, bin-width and AP transformations.


In [ ]:
import camb
from scipy.integrate import cumulative_trapezoid, simpson
from scipy.interpolate import PchipInterpolator, RectBivariateSpline, interp1d
from scipy.special import erf


THEORY_LINE_COLOR = "black"
THEORY_LINEWIDTH_PT = 0.85
THEORY_LINE_ZORDER = 0.8
THEORY_BIN_WIDTH_MAG = 0.5
THEORY_GRID_POINTS = 171
THEORY_MODEL_IDS = (
    "fixed_cosmo_astro", "fixed_cosmology", "no_pbh",
)
THEORY_MODEL_LABELS = (
    "Fixed SF", "Free SF", "No PBH",
)
THEORY_MODEL_LINESTYLES = ("-", "--", ":")
FIXED_COSMO_ASTRO_PBH_PARAMETERS = np.array([3.725892, -4.749689, -5.937677, -12.6013, -1.348524, -0.591189, 0.037844, -5.782382], dtype=float)
FIXED_COSMO_ASTRO_ASTROPHYSICS = np.array([-0.46, 0.9, -2, -2.2, 1.5, 12, 0.7, 2])
NO_PBH_ASTROPHYSICS_PARAMETERS = np.array([-0.468475, 2.826121, -2.800398, -2.362697, -1.49888, 12.032691, 0.60321, 1.527712], dtype=float)
FIXED_COSMOLOGY_IMAGE_PARAMETERS = np.array([3.989841, -3.338759, -9.393956, -8.475337, -1.825136, -1.368994, -1.240879, -8.102326, -0.434528, 2.425452, -2.810426, -2.569953, 1.176939, 11.738596, 1.131876, 1.817875], dtype=float)
THEORY_PARAMETER_ORDER = (
    "logM_PBH", "logf_PBH", "loglambda_E68", "loglambda_E9",
    "loglambda_E10", "loglambda_E11", "loglambda_E12", "loglambda_E13p5",
    "alphastar", "betastar", "epsilonstar_slope", "epsilonstar_icept",
    "Mc_slope", "Mc_icept", "sigma_MUV", "Q",
)
PANEL_SLOT_INDICES = (0, 1, 2, 4, 5, 6, 7)
LEGEND_SLOT_INDEX = 3
THEORY_LEGEND_Y_AXES = -0.02
LEGEND_MIN_VERTICAL_GAP_PT = 5.0

rho_crit = 2.77463e11
AST = 0.3222
aST = 0.707
pST = 0.3
deltaST = 1.686
kappaUV = 1.15e-28
invMpctoinvYear = 3.066067275784999e-07
c = 299792.458
h_HST = 0.7
Omega_m_HST = 0.3
deltaz = 0.5
sigma8_P18 = 0.8102
omega_cdm_P18 = 0.11933
omega_b_P18 = 0.02242
n_s_P18 = 0.9665
points, weights = np.polynomial.legendre.leggauss(25)
Mhalos = np.geomspace(1e8, 1e14, 1000)
sigma_cb_interp_func = None
BETA_PARAMETER_ROWS = np.array([
    [2.5, -1.70, -0.20],
    [3.8, -1.85, -0.11],
    [5.0, -1.91, -0.14],
    [5.9, -2.00, -0.20],
    [7.0, -2.05, -0.20],
    [8.0, -2.13, -0.15],
], dtype=float)
betainterp = PchipInterpolator(BETA_PARAMETER_ROWS[:, 0], BETA_PARAMETER_ROWS[:, 1])
dbetadMUVinterp = PchipInterpolator(BETA_PARAMETER_ROWS[:, 0], BETA_PARAMETER_ROWS[:, 2])




























def legacy_build_sigma_cb_interp(camb_results, zs_for_sigma):
    """
    预计算 sigma_cb(R, z) 的二维插值函数。
    
    CAMB get_sigmaR 的正确用法：
    - z_indices=None 时返回所有预计算 redshift 的结果，形状 (n_z, n_R)
    - z_indices=i 时返回第 i 个 redshift 的结果，形状 (n_R,)
    - redshifts 在 CAMB 内部按**降序**存储
    """
    R_arr = np.logspace(-2, 2.5, 200)  # Mpc 单位

    # CAMB 内部将 redshift 按降序排列
    # zs_for_sigma 传入时是升序的，CAMB 内部会反转
    # get_sigmaR 返回的 z 维度也是按 CAMB 内部降序排列的

    # 一次性获取所有 z 的 sigma(R)，返回 shape = (n_z, n_R)
    sigma_all = camb_results.get_sigmaR(
        R_arr, hubble_units=False,
        var1='delta_nonu', var2='delta_nonu'
    )
    # sigma_all 的 z 维度按 CAMB 内部顺序（降序）排列
    # 获取 CAMB 内部存储的 redshift 顺序
    # transfer_redshifts 是 CAMB 内部降序排列的 redshift 列表
    z_camb = camb_results.transfer_redshifts  # 降序

    # 反转为升序，方便插值
    z_ascending = np.array(z_camb[::-1])
    sigma_ascending = sigma_all[::-1, :]  # 对应反转 z 维度

    # 构建二维插值: log10(R) vs z -> sigma
    logR_arr = np.log10(R_arr)

    # 使用 RectBivariateSpline 进行二维插值
    # 注意：RectBivariateSpline 要求 x, y 严格递增
    interp_2d = RectBivariateSpline(z_ascending, logR_arr, sigma_ascending, kx=3, ky=3)

    def sigma_cb_func(R, z):
        """返回 sigma_cb(R, z)，R 单位 Mpc"""
        logR = np.log10(R)
        val = interp_2d(z, logR, grid=False)
        return float(val)

    return sigma_cb_func


def legacy_build_growth_interps(camb_results, pars):
    """
    构建:
    - D(z): scale-independent growth factor (归一化到 z=0 时 D=1)
    - f(z): growth rate f = d ln D / d ln a
    - H(z): Hubble parameter in 1/Mpc units
    """
    z_arr = np.linspace(0, 14, 500)
    h = pars.H0 / 100.0

    # 获取 sigma8(z=0) 用于归一化
    sigma8_0 = camb_results.get_sigmaR(
        8.0 / h, hubble_units=False,
        var1='delta_nonu', var2='delta_nonu'
    )
    # sigma8_0 shape = (n_z,)，z=0 是最后一个（CAMB 内部降序）
    # 取 z=0 对应的值
    sigma8_0_val = float(sigma8_0[-1])  # 最后一个是 z=0（降序中最小的 z）

    # 对每个 z 计算 D(z) = sigma_cb(R=8/h, z) / sigma_cb(R=8/h, 0)
    # 使用已经构建好的 sigma_cb_interp_func
    D_arr = np.zeros_like(z_arr)
    R8 = 8.0 / h
    for i, z in enumerate(z_arr):
        sig_z = sigma_cb_interp_func(R8, z)
        D_arr[i] = sig_z / sigma8_0_val if sigma8_0_val > 0 else 0.0

    # f(z) = - (1+z) / D * dD/dz，用数值微分
    f_arr = np.zeros_like(z_arr)
    for i in range(1, len(z_arr) - 1):
        dDdz = (D_arr[i + 1] - D_arr[i - 1]) / (z_arr[i + 1] - z_arr[i - 1])
        f_arr[i] = -(1 + z_arr[i]) / D_arr[i] * dDdz if D_arr[i] > 0 else 0.0
    f_arr[0] = f_arr[1]
    f_arr[-1] = f_arr[-2]

    # H(z) in 1/Mpc units: H_camb [km/s/Mpc] -> H [1/Mpc] = H_camb / c
    H_arr = np.array([camb_results.hubble_parameter(z) for z in z_arr])
    H_arr_invMpc = H_arr / c  # c = 299792.458 km/s

    growth_interp = interp1d(z_arr, D_arr, kind='cubic', fill_value='extrapolate')
    growthf_interp = interp1d(z_arr, f_arr, kind='cubic', fill_value='extrapolate')
    hubble_interp = interp1d(z_arr, H_arr_invMpc, kind='cubic', fill_value='extrapolate')

    return growth_interp, growthf_interp, hubble_interp


def beta_average(z, MUV):
    if MUV < -19.5:
        return dbetadMUVinterp(z) * (MUV + 19.5) + betainterp(z)
    return (
        (betainterp(z) + 2.33)
        * np.exp(dbetadMUVinterp(z) * (MUV + 19.5) / (betainterp(z) + 2.33))
        - 2.33
    )


def AUV(z, MUV):
    if z < 2.5 or z > 8.0:
        return 0.0
    sigma_beta = 0.34
    return max(
        0.0,
        4.54
        + 0.2 * np.log(10.0) * (2.07**2) * (sigma_beta**2)
        + 2.07 * beta_average(z, MUV),
    )


def published_frame_mapping(z, observed_magnitudes, observed_bin_width, Omega_m, h):
    observed_magnitudes = np.asarray(observed_magnitudes, dtype=float)
    dust_at_center = np.array([AUV(z, value) for value in observed_magnitudes])
    dust_at_faint_edge = np.array([
        AUV(z, value + observed_bin_width / 2.0) for value in observed_magnitudes
    ])
    dust_at_bright_edge = np.array([
        AUV(z, value - observed_bin_width / 2.0) for value in observed_magnitudes
    ])
    model_bin_width = (
        observed_bin_width - dust_at_faint_edge + dust_at_bright_edge
    )
    if np.any(model_bin_width <= 0.0):
        raise FloatingPointError(f"Non-positive dust-corrected bin width at z={z}")

    model_distance = rcomoving(z, Omega_m, h)
    published_distance = rcomoving(z, Omega_m_HST, h_HST)
    AP_delta_M = 5.0 * np.log10(model_distance / published_distance)
    AP_volume_ratio = (
        rcomoving(z + deltaz, Omega_m_HST, h_HST) ** 3
        - rcomoving(z - deltaz, Omega_m_HST, h_HST) ** 3
    ) / (
        rcomoving(z + deltaz, Omega_m, h) ** 3
        - rcomoving(z - deltaz, Omega_m, h) ** 3
    )
    if not np.isfinite(AP_volume_ratio) or AP_volume_ratio <= 0.0:
        raise FloatingPointError(f"Invalid AP volume ratio at z={z}")

    model_magnitudes = observed_magnitudes - dust_at_center - AP_delta_M
    inverse_density_factor = (
        model_bin_width / observed_bin_width / AP_volume_ratio
    )
    return {
        "A_UV": dust_at_center,
        "M_UV_model": model_magnitudes,
        "model_bin_width": model_bin_width,
        "AP_delta_M": float(AP_delta_M),
        "AP_volume_ratio": float(AP_volume_ratio),
        "inverse_density_factor": inverse_density_factor,
    }


def build_fixed_planck18_cosmology(density_var="delta_nonu"):
    pars = camb.CAMBparams()
    pars.set_cosmology(
        H0=None,
        cosmomc_theta=1.041638 / 100.0,
        ombh2=omega_b_P18,
        omch2=omega_cdm_P18,
        tau=0.0544,
        mnu=0.06,
        nnu=3.044,
        num_massive_neutrinos=1,
        **({"theta_H0_range": [10.0, 150.0]} if density_var == "delta_tot" else {}),
    )
    pars.InitPower.set_params(ns=n_s_P18, As=2e-9, pivot_scalar=0.05)
    z_for_sigma = np.linspace(0.0, 13.0, 60)
    pars.set_matter_power(
        redshifts=sorted(z_for_sigma, reverse=True),
        kmax=5.0,
        nonlinear=False,
    )
    pars.WantTransfer = True
    results = camb.get_results(pars)
    h_temporary = results.Params.H0 / 100.0
    sigma8_values = results.get_sigmaR(
        8.0 / h_temporary,
        hubble_units=False,
        var1=density_var,
        var2=density_var,
    )
    sigma8_current = float(sigma8_values[-1])
    if not np.isfinite(sigma8_current) or sigma8_current <= 0.0:
        raise FloatingPointError("CAMB returned an invalid sigma8 normalization")
    calibrated_As = pars.InitPower.As * (sigma8_P18 / sigma8_current) ** 2
    pars.InitPower.set_params(ns=n_s_P18, As=calibrated_As, pivot_scalar=0.05)
    results = camb.get_results(pars)
    h = results.Params.H0 / 100.0
    Omega_m = (
        results.get_Omega("cdm")
        + results.get_Omega("baryon")
        + results.get_Omega("nu")
    )
    Omega_b = results.get_Omega("baryon")
    return results, pars, z_for_sigma, h, Omega_b, Omega_m, calibrated_As


def HMF_no_pbh(z, M, h, Omega_m):
    rhoM = h**2 * Omega_m * rho_crit
    deltaM = 100.0

    def sigma_standard(mass):
        radius = (3.0 * mass / (4.0 * np.pi * rhoM)) ** (1.0 / 3.0)
        return float(sigma_cb_interp_func(radius, z))

    sigma = sigma_standard(M)
    dsigmadM = (sigma_standard(M + deltaM) - sigma_standard(M - deltaM)) / (2.0 * deltaM)
    return (
        -AST
        * np.sqrt(2.0 * aST / np.pi)
        * (1.0 + (sigma**2 / (aST * deltaST**2)) ** pST)
        * (deltaST / sigma)
        * np.exp(-aST * deltaST**2 / (2.0 * sigma**2))
        * (rhoM / (M * sigma))
        * dsigmadM
    )


def MUV_from_Mh_no_pbh(
    growth_interp, growthf_interp, hubble_interp,
    z, Mh, h, Omega_m,
    alphastar, betastar, epsilonstar_slope, epsilonstar_icept,
    Mc_slope, Mc_icept, Q,
):
    Mh = float(Mh)
    epsilonstar = 10.0 ** (
        epsilonstar_slope * np.log10((1.0 + z) / (1.0 + 6.0)) + epsilonstar_icept
    )
    Mc = 10.0 ** (Mc_slope * np.log10((1.0 + z) / (1.0 + 6.0)) + Mc_icept)
    rhoM = h**2 * Omega_m * rho_crit
    radius1 = (3.0 * Mh / Q / (4.0 * np.pi * rhoM)) ** (1.0 / 3.0)
    radius2 = (3.0 * Mh / (4.0 * np.pi * rhoM)) ** (1.0 / 3.0)
    sigma1 = float(sigma_cb_interp_func(radius1, z))
    sigma2 = float(sigma_cb_interp_func(radius2, z))
    sigma_difference = sigma1**2 - sigma2**2
    if not np.isfinite(sigma_difference) or sigma_difference <= 0.0:
        raise FloatingPointError(f"Invalid no-PBH sigma difference at z={z}, Mh={Mh}")
    functionf = 1.0 / np.sqrt(sigma_difference)
    Dz = float(growth_interp(z))
    fz = float(growthf_interp(z))
    Hz = float(hubble_interp(z))
    dgrowthdz = -fz * Dz / (1.0 + z)
    Mhdot = (
        -(1.0 + z) * Hz * invMpctoinvYear * 1.686 * np.sqrt(2.0 / np.pi)
        * Mh * functionf * dgrowthdz / Dz**2
    )
    stellar_uv_luminosity = (
        epsilonstar * Mhdot
        / ((Mh / Mc) ** alphastar + (Mh / Mc) ** betastar)
        / kappaUV
    )
    if not np.isfinite(stellar_uv_luminosity) or stellar_uv_luminosity <= 0.0:
        raise FloatingPointError(f"Invalid stellar UV luminosity at z={z}, Mh={Mh}")
    return -2.5 * np.log10(stellar_uv_luminosity) + 51.63


def compute_uvlf_theory_curves(model_parameters, redshifts, magnitude_grid):
    global sigma_cb_interp_func
    redshifts = np.asarray(redshifts, dtype=float)
    magnitude_grid = np.asarray(magnitude_grid, dtype=float)
    model_parameters = list(model_parameters)
    if len(model_parameters) != 3:
        raise ValueError("Exactly three theory model specifications are required")
    if tuple(spec["model_id"] for spec in model_parameters) != THEORY_MODEL_IDS:
        raise ValueError("Theory model order does not match the declared model IDs")
    if set(redshifts.tolist()) != set(PANELS):
        raise ValueError("redshifts must match the seven displayed panels")
    if magnitude_grid.ndim != 1 or magnitude_grid.size < 2:
        raise ValueError("magnitude_grid must be one-dimensional")
    if not np.all(np.diff(magnitude_grid) > 0):
        raise ValueError("magnitude_grid must be strictly increasing")

    

    frames = []
    panel_diagnostics = []
    for spec in model_parameters:
        enabled = bool(spec['pbh_enabled'])
        results, pars, z_for_sigma, h, Omega_b, Omega_m, calibrated_As = build_fixed_planck18_cosmology(
            'delta_tot' if enabled else 'delta_nonu')
        sigma_builder = build_sigma_cb_interp if enabled else legacy_build_sigma_cb_interp
        growth_builder = build_growth_interps if enabled else legacy_build_growth_interps
        sigma_cb_interp_func = sigma_builder(results, z_for_sigma)
        growth_interp, growthf_interp, hubble_interp = growth_builder(results, pars)
        param = np.asarray(spec["parameters"], dtype=float)
        if param.shape != (16,) or not np.isfinite(param).all():
            raise ValueError(f"{spec['model_id']} must contain sixteen finite values")
        (
            logM_PBH, logf_PBH, loglambda_E68, loglambda_E9,
            loglambda_E10, loglambda_E11, loglambda_E12, loglambda_E13p5,
            alphastar, betastar, epsilonstar_slope, epsilonstar_icept,
            Mc_slope, Mc_icept, sigma_MUV, Q,
        ) = param
        pbh_enabled = bool(spec["pbh_enabled"])
        if pbh_enabled:
            M_BH_of_z, _, _ = build_MBH_growth_factor_from_camb(
                camb_results=results,
                loglambda_E68=loglambda_E68,
                loglambda_E9=loglambda_E9,
                loglambda_E10=loglambda_E10,
                loglambda_E11=loglambda_E11,
                loglambda_E12=loglambda_E12,
                loglambda_E13p5=loglambda_E13p5,
                f_duty=1.0,
                t_Sal=0.045,
                z_merge=20.0,
                z_final=6.0,
                n_points=5000,
            )
            physical_M_PBH = 10.0**logM_PBH
            physical_f_PBH = 10.0**logf_PBH
        else:
            M_BH_of_z = None
            physical_M_PBH = 0.0
            physical_f_PBH = 0.0

        for z in redshifts:
            if pbh_enabled:
                HMFs = np.array([
                    HMF(z, mass, logM_PBH, logf_PBH, h, Omega_b, Omega_m)
                    for mass in Mhalos
                ])
                MUV_averages = np.array([
                    MUV_from_Mh(
                        growth_interp, growthf_interp, hubble_interp,
                        z, mass, logM_PBH, logf_PBH, h, Omega_b, Omega_m,
                        loglambda_E68, loglambda_E9, loglambda_E10,
                        loglambda_E11, loglambda_E12,
                        alphastar, betastar, epsilonstar_slope, epsilonstar_icept,
                        Mc_slope, Mc_icept, Q, M_BH_of_z,
                    )
                    for mass in Mhalos
                ])
            else:
                HMFs = np.array([
                    HMF_no_pbh(z, mass, h, Omega_m) for mass in Mhalos
                ])
                MUV_averages = np.array([
                    MUV_from_Mh_no_pbh(
                        growth_interp, growthf_interp, hubble_interp,
                        z, mass, h, Omega_m,
                        alphastar, betastar, epsilonstar_slope, epsilonstar_icept,
                        Mc_slope, Mc_icept, Q,
                    )
                    for mass in Mhalos
                ])
            if not np.isfinite(HMFs).all() or not np.isfinite(MUV_averages).all():
                raise FloatingPointError(
                    f"Non-finite halo calculation for {spec['model_id']} at z={z}"
                )
            frame_mapping = published_frame_mapping(
                z, magnitude_grid, THEORY_BIN_WIDTH_MAG, Omega_m, h,
            )
            model_bin_width = frame_mapping["model_bin_width"]
            bin_probabilities = first_integrand(
                frame_mapping["M_UV_model"][:, None],
                model_bin_width[:, None],
                MUV_averages[None, :],
                sigma_MUV,
            )
            phi_model = simpson(
                HMFs[None, :] * bin_probabilities / model_bin_width[:, None],
                x=Mhalos,
                axis=1,
            )
            if not np.isfinite(phi_model).all() or np.any(phi_model <= 0.0):
                raise FloatingPointError(
                    f"Theory UVLF is not finite and positive for {spec['model_id']} at z={z}"
                )
            phi_observed = phi_model * frame_mapping["inverse_density_factor"]
            if not np.isfinite(phi_observed).all() or np.any(phi_observed <= 0.0):
                raise FloatingPointError(
                    f"Observed-frame UVLF is invalid for {spec['model_id']} at z={z}"
                )
            frames.append(pd.DataFrame({
                "model_id": spec["model_id"],
                "model_label": spec["label"],
                "line_style": spec["line_style"],
                "pbh_enabled": pbh_enabled,
                "M_PBH": physical_M_PBH,
                "f_PBH": physical_f_PBH,
                "redshift": float(z),
                "M_UV": magnitude_grid,
                "A_UV": frame_mapping["A_UV"],
                "M_UV_model": frame_mapping["M_UV_model"],
                "model_bin_width": model_bin_width,
                "AP_delta_M": frame_mapping["AP_delta_M"],
                "AP_volume_ratio": frame_mapping["AP_volume_ratio"],
                "phi_model": phi_model,
                "phi_observed": phi_observed,
            }))
            panel_diagnostics.append({
                "model_id": spec["model_id"],
                "redshift": float(z),
                "MBH_growth_factor": float(M_BH_of_z(z)) if pbh_enabled else 0.0,
                "A_UV_max": float(frame_mapping["A_UV"].max()),
                "AP_delta_M": frame_mapping["AP_delta_M"],
                "AP_volume_ratio": frame_mapping["AP_volume_ratio"],
                "phi_observed_min": float(phi_observed.min()),
                "phi_observed_max": float(phi_observed.max()),
            })

    theory_frame = pd.concat(frames, ignore_index=True)
    diagnostics = {
        "H0": float(results.Params.H0),
        "h": float(h),
        "Omega_m": float(Omega_m),
        "Omega_b": float(Omega_b),
        "As": float(calibrated_As),
        "panels": panel_diagnostics,
    }
    return theory_frame, diagnostics

k_sigma = np.logspace(-5, 1, 1000)
k_sigma_factor = k_sigma**2 / (2.0 * np.pi**2)
def D_p(z, Omega_m, Omega_b):
    r = (Omega_m - Omega_b) / Omega_m
    a = 1 / 4 * ((1 + 24 * r) ** 0.5 - 1)
    d2 = (1 + 3 * r * 3401 / (2 * (1 + z) * a)) ** a
    return d2

def p_PBH(logM_PBH, logf_PBH, Omega_m, Omega_b, h):
    return (10**logM_PBH * 10**logf_PBH) / (
        (Omega_m - Omega_b) * rho_crit * h**2
    )

def w_f(k, R):
    x = k * R
    return 3 * (np.sin(x) - x * np.cos(x)) / x**3

def sigma_PBH(R, z, logM_PBH, logf_PBH, Omega_m, Omega_b, h):
    """
    使用全局变量 sigma_cb_interp_func 来获取 sigma_cb(R, z)
    """
    integrand = (
        p_PBH(logM_PBH, logf_PBH, Omega_m, Omega_b, h)
        * D_p(z, Omega_m, Omega_b) ** 2
        * w_f(k_sigma, R) ** 2
        * k_sigma_factor
    )
    sigma_cb_val = sigma_cb_interp_func(R, z)
    return np.sqrt(simpson(integrand, x=k_sigma) + sigma_cb_val**2)

def lambda_of_z_from_loglambda(
    z,
    loglambda_E68,
    loglambda_E9,
    loglambda_E10,
    loglambda_E11,
    loglambda_E12,
    loglambda_E13p5,
):
    """
    分段定义 Eddington ratio：

    6.0   <= z < 8.5    : loglambda_E68
    8.5   <= z < 9.5    : loglambda_E9
    9.5   <= z < 10.5   : loglambda_E10
    10.5  <= z < 11.5   : loglambda_E11
    11.5  <= z < 13.5   : loglambda_E12
    13.5  <= z <= 1000  : loglambda_E13p5
    """
    z = np.asarray(z)

    loglam = np.zeros_like(z, dtype=float)

    loglam[(z >= 6.0) & (z < 8.5)] = loglambda_E68
    loglam[(z >= 8.5) & (z < 9.5)] = loglambda_E9
    loglam[(z >= 9.5) & (z < 10.5)] = loglambda_E10
    loglam[(z >= 10.5) & (z < 11.5)] = loglambda_E11
    loglam[(z >= 11.5) & (z < 13.5)] = loglambda_E12
    loglam[(z >= 13.5) & (z <= 1000.0)] = loglambda_E13p5

    return 10.0 ** loglam

def camb_Hz_Gyr_inv(camb_results, z):
    """
    CAMB H(z): km/s/Mpc -> Gyr^-1
    """

    Hz_km_s_Mpc = camb_results.hubble_parameter(z)

    return Hz_km_s_Mpc * 1.022712165e-3

def build_MBH_growth_factor_from_camb(
    camb_results,
    loglambda_E68,
    loglambda_E9,
    loglambda_E10,
    loglambda_E11,
    loglambda_E12,
    loglambda_E13p5,
    f_duty=1.0,
    t_Sal=0.045,
    z_merge=20.0,
    z_final=6.0,
    n_points=5000,
):
    """返回 z=20 合并后单个大黑洞的无量纲质量增长因子。"""
    z_bh_grid = np.linspace(z_merge, z_final, n_points)

    lambda_grid = lambda_of_z_from_loglambda(
        z_bh_grid,
        loglambda_E68,
        loglambda_E9,
        loglambda_E10,
        loglambda_E11,
        loglambda_E12,
        loglambda_E13p5,
    )

    H_grid = camb_Hz_Gyr_inv(camb_results, z_bh_grid)

    if np.any(H_grid <= 0.0):
        raise ValueError("H(z) contains non-positive values.")

    integrand = (
        f_duty
        * lambda_grid
        / (t_Sal * (1.0 + z_bh_grid) * H_grid)
    )

    cumulative_integral = cumulative_trapezoid(
        integrand,
        z_bh_grid,
        initial=0.0,
    )

    ln_growth = -cumulative_integral

    if np.any(ln_growth > 700):
        raise OverflowError("Black hole growth exponent too large.")

    growth_factor_grid = np.exp(ln_growth)

    if np.any(~np.isfinite(growth_factor_grid)):
        raise FloatingPointError("M_BH growth factor contains non-finite values.")

    growth_factor_of_z = interp1d(
        z_bh_grid[::-1],
        growth_factor_grid[::-1],
        kind="linear",
        bounds_error=True,
    )

    return growth_factor_of_z, z_bh_grid, growth_factor_grid

def merged_pbh_mass(Mh, logf_PBH):
    """z=20 时晕内所有 PBH 瞬时合并后的总质量。"""
    return float(Mh) * np.power(10.0, float(logf_PBH))

def L1_PBH(z,
    loglambda_E,
    logf_PBH,
    Mh,
    MBH_growth_factor_of_z,):
    M_BH_z = (
        merged_pbh_mass(Mh, logf_PBH)
        * float(MBH_growth_factor_of_z(z))
    )
    if not np.isfinite(M_BH_z):
        raise FloatingPointError("Merged M_BH contains a non-finite value.")
    L_b = 3.28e4 * 3.846e33 * np.power(10, loglambda_E) * M_BH_z
    if not np.isfinite(L_b):
        raise FloatingPointError("Merged black-hole luminosity is non-finite.")
    return (
        L_b
        / (
            1.862 * (float(L_b) / 3.846e43) ** (-0.361)
            + 4.870 * (L_b / 3.846e43) ** (-0.0063)
        )
        * 4.8367e-16
    )

def HMF(z, M, logM_PBH, logf_PBH, h, Omega_b, Omega_m):
    rhoM = np.power(h, 2) * Omega_m * rho_crit
    deltaM = 100
    sigma = sigma_PBH(
        np.power(3.0 * M / (4.0 * np.pi * rhoM), 1.0 / 3),
        z, logM_PBH, logf_PBH, Omega_m, Omega_b, h,
    )
    dsigmadM = (
        sigma_PBH(
            np.power(3.0 * (M + deltaM) / (4.0 * np.pi * rhoM), 1.0 / 3),
            z, logM_PBH, logf_PBH, Omega_m, Omega_b, h,
        )
        - sigma_PBH(
            np.power(3.0 * (M - deltaM) / (4.0 * np.pi * rhoM), 1.0 / 3),
            z, logM_PBH, logf_PBH, Omega_m, Omega_b, h,
        )
    ) / (2.0 * deltaM)
    return (
        -AST
        * np.sqrt(2.0 * aST / np.pi)
        * (
            1.0
            + np.power(
                np.power(sigma, 2) / (aST * np.power(deltaST, 2)),
                pST,
            )
        )
        * (deltaST / sigma)
        * np.exp(-aST * np.power(deltaST, 2) / (2.0 * np.power(sigma, 2)))
        * (rhoM / (M * sigma))
        * dsigmadM
    )

def MUV_from_Mh(
    growth_interp, growthf_interp, hubble_interp,
    z, Mh, logM_PBH, logf_PBH, h, Omega_b, Omega_m,
    loglambda_E68, loglambda_E9, loglambda_E10, loglambda_E11, loglambda_E12,
    alphastar, betastar, epsilonstar_slope, epsilonstar_icept,
    Mc_slope, Mc_icept, Q, MBH_growth_factor_of_z,
):
    z_float = float(z)

    if z_float in [6.0, 7.0, 8.0]:
        loglambda_E = float(loglambda_E68)
    elif z_float == 9.0:
        loglambda_E = float(loglambda_E9)    
    elif z_float == 10.0:
        loglambda_E = float(loglambda_E10)
    elif z_float == 11.0:
        loglambda_E = float(loglambda_E11)
    elif z_float == 12.5:
        loglambda_E = float(loglambda_E12)
    else:
        raise ValueError(
            f"Redshift z={z} is not in the defined ranges (6, 7, 8, 9, 10, 11, 12.5)."
        )

    Mh = float(Mh)
    logM_PBH = float(logM_PBH)
    epsilonstar = 10 ** (
        epsilonstar_slope * np.log10((1 + z) / (1 + 6)) + epsilonstar_icept
    )
    Mc = 10 ** (Mc_slope * np.log10((1 + z) / (1 + 6)) + Mc_icept)

    rhoM = np.power(h, 2) * Omega_m * rho_crit

    sigma1 = sigma_PBH(
        np.power(3.0 * Mh / Q / (4.0 * np.pi * rhoM), 1.0 / 3),
        z, logM_PBH, logf_PBH, Omega_m, Omega_b, h,
    )
    sigma2 = sigma_PBH(
        np.power(3.0 * Mh / (4.0 * np.pi * rhoM), 1.0 / 3),
        z, logM_PBH, logf_PBH, Omega_m, Omega_b, h,
    )

    functionf = 1.0 / np.sqrt(sigma1**2 - sigma2**2)

    # 使用预计算的插值
    Dz = growth_interp(z)
    fz = growthf_interp(z)
    Hz = hubble_interp(z)  # 单位: 1/Mpc

    dgrowthdz = -fz * Dz / (1.0 + z)

    Mhdot = (
        -(1 + z)
        * Hz
        * invMpctoinvYear
        * 1.686
        * np.sqrt(2.0 / np.pi)
        * Mh
        * functionf
        * dgrowthdz
        / Dz**2
    )

    return (
        -2.5
        * np.log10(
            epsilonstar
            * Mhdot
            / ((Mh / Mc) ** alphastar + (Mh / Mc) ** betastar)
            / kappaUV
            + L1_PBH(z,
    loglambda_E,
    logf_PBH,
    Mh,
    MBH_growth_factor_of_z,)
        )
        + 51.63
    )

def first_integrand(MUV, width, MUV_av, sigma_MUV):
    return 0.5 * (
        erf((MUV_av - MUV + width / 2.0) / (sigma_MUV * np.sqrt(2)))
        - erf((MUV_av - MUV - width / 2.0) / (sigma_MUV * np.sqrt(2)))
    )

def rcomoving(z, Omega_m, h):
    return (
        c
        * integrator(
            lambda x: 1 / np.sqrt(Omega_m * np.power(1 + x, 3) + 1.0 - Omega_m),
            0.0,
            z,
        )
        / (100.0 * h)
    )

def integrator(f, a, b):
    sub = (b - a) / 2.0
    add = (b + a) / 2.0
    if sub == 0:
        return 0.0
    return sub * np.dot(f(sub * points + add), weights)

def build_sigma_cb_interp(camb_results, zs_for_sigma):
    """
    预计算 sigma_cb(R, z) 的二维插值函数。
    
    CAMB get_sigmaR 的正确用法：
    - z_indices=None 时返回所有预计算 redshift 的结果，形状 (n_z, n_R)
    - z_indices=i 时返回第 i 个 redshift 的结果，形状 (n_R,)
    - redshifts 在 CAMB 内部按**降序**存储
    """
    R_arr = np.logspace(-2, 2.5, 200)  # Mpc 单位

    # CAMB 内部将 redshift 按降序排列
    # zs_for_sigma 传入时是升序的，CAMB 内部会反转
    # get_sigmaR 返回的 z 维度也是按 CAMB 内部降序排列的

    # 一次性获取所有 z 的 sigma(R)，返回 shape = (n_z, n_R)
    sigma_all = camb_results.get_sigmaR(
        R_arr, hubble_units=False,
        var1='delta_tot', var2='delta_tot'
    )
    # sigma_all 的 z 维度按 CAMB 内部顺序（降序）排列
    # 获取 CAMB 内部存储的 redshift 顺序
    # transfer_redshifts 是 CAMB 内部降序排列的 redshift 列表
    z_camb = camb_results.transfer_redshifts  # 降序

    # 反转为升序，方便插值
    z_ascending = np.array(z_camb[::-1])
    sigma_ascending = sigma_all[::-1, :]  # 对应反转 z 维度

    # 构建二维插值: log10(R) vs z -> sigma
    logR_arr = np.log10(R_arr)

    # 使用 RectBivariateSpline 进行二维插值
    # 注意：RectBivariateSpline 要求 x, y 严格递增
    interp_2d = RectBivariateSpline(z_ascending, logR_arr, sigma_ascending, kx=3, ky=3)

    def sigma_cb_func(R, z):
        """返回 sigma_cb(R, z)，R 单位 Mpc"""
        logR = np.log10(R)
        val = interp_2d(z, logR, grid=False)
        return float(val)

    return sigma_cb_func

def build_growth_interps(camb_results, pars):
    """
    构建:
    - D(z): scale-independent growth factor (归一化到 z=0 时 D=1)
    - f(z): growth rate f = d ln D / d ln a
    - H(z): Hubble parameter in 1/Mpc units
    """
    z_arr = np.linspace(0, 14, 500)
    h = pars.H0 / 100.0

    # 获取 sigma8(z=0) 用于归一化
    sigma8_0 = camb_results.get_sigmaR(
        8.0 / h, hubble_units=False,
        var1='delta_tot', var2='delta_tot'
    )
    # sigma8_0 shape = (n_z,)，z=0 是最后一个（CAMB 内部降序）
    # 取 z=0 对应的值
    sigma8_0_val = float(sigma8_0[-1])  # 最后一个是 z=0（降序中最小的 z）

    # 对每个 z 计算 D(z) = sigma_cb(R=8/h, z) / sigma_cb(R=8/h, 0)
    # 使用已经构建好的 sigma_cb_interp_func
    D_arr = np.zeros_like(z_arr)
    R8 = 8.0 / h
    for i, z in enumerate(z_arr):
        sig_z = sigma_cb_interp_func(R8, z)
        D_arr[i] = sig_z / sigma8_0_val if sigma8_0_val > 0 else 0.0

    # f(z) = - (1+z) / D * dD/dz，用数值微分
    f_arr = np.zeros_like(z_arr)
    for i in range(1, len(z_arr) - 1):
        dDdz = (D_arr[i + 1] - D_arr[i - 1]) / (z_arr[i + 1] - z_arr[i - 1])
        f_arr[i] = -(1 + z_arr[i]) / D_arr[i] * dDdz if D_arr[i] > 0 else 0.0
    f_arr[0] = f_arr[1]
    f_arr[-1] = f_arr[-2]

    # H(z) in 1/Mpc units: H_camb [km/s/Mpc] -> H [1/Mpc] = H_camb / c
    H_arr = np.array([camb_results.hubble_parameter(z) for z in z_arr])
    H_arr_invMpc = H_arr / c  # c = 299792.458 km/s

    growth_interp = interp1d(z_arr, D_arr, kind='cubic', fill_value='extrapolate')
    growthf_interp = interp1d(z_arr, f_arr, kind='cubic', fill_value='extrapolate')
    hubble_interp = interp1d(z_arr, H_arr_invMpc, kind='cubic', fill_value='extrapolate')

    return growth_interp, growthf_interp, hubble_interp

MODEL_SOURCE_SHA256 = {'emcee_UVLF_only8_merged_z20.py': '6e7cac582577d80c41c38fe263b8a086b9742a3b0fc64dea0c440d0e29f8e8cb', 'emcee_UVLF_P18_a16_merged_z20.py': '994e08bb2de4fa6ad1ecbbad44e818d5308b7a4cbbffb150fd14172dee45bdde'}

DISPLAY_MODEL_IDS = ("fixed_cosmology", "no_pbh")
DISPLAY_MODEL_LABELS = ("With PBH", "No PBH")
DISPLAY_MODEL_LINESTYLES = ("-", ":")


In [ ]:
# Display-only revision: reuse the previously calculated observed-frame curves.
cached_theory = pd.read_csv("UVLF_full_SF_theory.txt", sep="\t", comment="#")
theory_curve_data = cached_theory.loc[cached_theory.model_id.isin(DISPLAY_MODEL_IDS)].copy()
theory_curve_data.loc[theory_curve_data.model_id.eq("fixed_cosmology"), "model_label"] = "With PBH"
theory_curve_data.loc[theory_curve_data.model_id.eq("fixed_cosmology"), "line_style"] = "-"
assert len(theory_curve_data) == 2 * len(PANELS) * THEORY_GRID_POINTS
assert set(theory_curve_data.model_id) == set(DISPLAY_MODEL_IDS)
assert not theory_curve_data.duplicated(["model_id", "redshift", "M_UV"]).any()
assert np.isfinite(theory_curve_data[["M_UV", "phi_observed"]]).all().all()
assert theory_curve_data.phi_observed.gt(0).all()
THEORY_DATA_PATH = Path(f"{OUTPUT_STEM.name}_theory.txt")
THEORY_DATA_PATH.write_text("# With PBH [solid]; No PBH [dotted]. Values unchanged from previous figure.\n", encoding="utf-8")
theory_curve_data.to_csv(THEORY_DATA_PATH, sep="\t", index=False, mode="a", float_format="%.10e")
caption = Path("UVLF_full_SF_caption.tex").read_text(encoding="utf-8")
caption = caption.replace("all three observed-frame", "both observed-frame")
start = caption.index("Three black curves compare")
end = caption.index("The dotted No PBH curve", start)
caption = caption[:start] + (
    "Two black curves compare the theoretical UV luminosity functions. "
    "The solid With PBH curve uses fixed Planck-2018 cosmology and the supplied "
    "16-parameter MAP vector with free astrophysical parameters. Halo PBHs merge "
    "instantaneously at $z=20$ and subsequently grow through accretion. "
    "The PBH model uses total-matter fluctuations; the No PBH reference retains "
    "its existing non-neutrino-matter convention. "
) + caption[end:]
caption = caption.replace("All three curves", "Both curves")
Path(f"{OUTPUT_STEM.name}_caption.tex").write_text(caption, encoding="utf-8")
print("PASS: only With PBH and No PBH selected; cached numerical curves unchanged.")


## Seven-panel figure

Points are never connected. Detection rows display the tabulated asymmetric uncertainty magnitudes. In-range one-sided limits use unfilled 0.5 pt line arrows drawn directly on the corresponding data/error-bar x coordinate and driven by `limit_type`; no filled Matplotlib caret is used. The vertical error-bar arm in the arrow direction is suppressed to avoid overlap, whereas any reported arm in the opposite direction remains visible. When a source reports a one-sided limit without a second uncertainty magnitude, a zero-length drawing extent is retained only for the numerical error-bar layer and is not treated as a measured uncertainty. Every central marker has a 1.0 mm transformed minimum extent. Colored comparison markers use redundant shapes and transparent interiors so error bars remain visible through them, while the primary Bouwens and Donnan measurements retain emphasis through black fill, full opacity, heavier error bars, and the highest drawing order.

Boundary conventions preserve the fixed axes without changing any stored ledger value. The single Harikane lower-limit row at z=7, M_UV=-23.7, phi=2.3e-9 is excluded from display because its central numerical anchor lies below the fixed y-axis range. Four Chemerynska detections centered exactly at the right boundary remain plotted; only the positive M_UV-error segment beyond the frame is clipped, without arrows or relocation. Four detection intervals that extend to <=0 remain clipped by the fixed logarithmic lower boundary. The one-row display exclusion and all clipping conventions must also be described in the manuscript caption.

In [ ]:
# Stable, ledger-keyed visual encodings. Primary styles are deliberately dominant.
from matplotlib.markers import MarkerStyle


def transformed_marker_extent_points(marker, markersize):
    marker_style = MarkerStyle(marker)
    path = marker_style.get_path().transformed(marker_style.get_transform())
    bounds = path.get_extents()
    return bounds.width * markersize, bounds.height * markersize


def transformed_arrow_patch_extent_points(arrow_patch, figure):
    path = arrow_patch.get_path().transformed(arrow_patch.get_transform())
    bounds = path.get_extents()
    points_per_pixel = 72.0 / figure.dpi
    return bounds.width * points_per_pixel, bounds.height * points_per_pixel


def marker_size_for_extent(marker, target_extent_pt=CENTRAL_MARKER_EXTENT_PT):
    width, height = transformed_marker_extent_points(marker, 1.0)
    return target_extent_pt / min(width, height)


PRIMARY_STYLES = {
    "bouwens2021_primary": dict(marker="o", color="#000000", mfc="#000000", mec="#000000", ms=marker_size_for_extent("o"), elinewidth=0.75, capsize=1.5, alpha=1.0, zorder=10),
    "donnan2024_primary": dict(marker="s", color="#000000", mfc="#000000", mec="#000000", ms=marker_size_for_extent("s"), elinewidth=0.75, capsize=1.5, alpha=1.0, zorder=10),
}
COMPARISON_MARKERS = (
    "D", "^", "v", "P", "X", "h", "<", ">", "*",
    "p", "H", "d", "1", "2", "3", "4",
)
COMPARISON_COLORS = (
    "#0072B2", "#D55E00", "#009E73", "#882255",
    "#332288", "#117733", "#AA4499", "#8C510A",
)

plotted_dataset_ids = tuple(sorted(data.loc[data["plot_flag"], "dataset_id"].unique()))
comparison_dataset_ids = tuple(dataset_id for dataset_id in plotted_dataset_ids if dataset_id not in PRIMARY_STYLES)
STYLE_BY_DATASET = {dataset_id: style.copy() for dataset_id, style in PRIMARY_STYLES.items()}
for index, dataset_id in enumerate(comparison_dataset_ids):
    color = COMPARISON_COLORS[index % len(COMPARISON_COLORS)]
    STYLE_BY_DATASET[dataset_id] = dict(
        marker=COMPARISON_MARKERS[index % len(COMPARISON_MARKERS)],
        color=color, mfc="none", mec=color, ms=marker_size_for_extent(COMPARISON_MARKERS[index % len(COMPARISON_MARKERS)]),
        elinewidth=0.5, capsize=1.5, alpha=0.68, zorder=3,
    )
STYLE_BY_DATASET["harikane2025_spec"]["marker"] = "8"
STYLE_BY_DATASET["harikane2025_spec"]["ms"] = marker_size_for_extent("8")
STYLE_BY_DATASET["curtislake2026_jades_dr4_deep"]["marker"] = "h"
STYLE_BY_DATASET["curtislake2026_jades_dr4_deep"]["ms"] = marker_size_for_extent("h")

REFERENCE_SUFFIXES = {
    "curtislake2026_jades_dr4_deep": " D",
    "curtislake2026_jades_dr4_medium": " M",
    "franco2025_cosmosweb_f814w": " 814",
    "franco2025_cosmosweb_f115w": " 115",
    "franco2025_cosmosweb_f150w": " 150",
    "rojasruiz2025_borgjwst_go1747": " 1747",
    "rojasruiz2025_borgjwst_go2426": " 2426",
    "whitler2026_jades_f115w": " 115",
    "whitler2026_jades_f150w": " 150",
}
REFERENCE_ABBREVIATIONS = {
    "Chemerynska": "Chem.",
    "Curtis-Lake": "C.-Lake",
    "Rojas-Ruiz": "R.-Ruiz",
}

def compact_reference_label(dataset_id, frame=data):
    row = frame.loc[frame["plot_flag"] & frame["dataset_id"].eq(dataset_id)].iloc[0]
    surname = str(row["reference"]).split(" et al.")[0]
    surname = REFERENCE_ABBREVIATIONS.get(surname, surname)
    return f"{surname}+{str(int(row['year']))[-2:]}{REFERENCE_SUFFIXES.get(dataset_id, '')}"

assert set(STYLE_BY_DATASET) == set(plotted_dataset_ids)
assert all(STYLE_BY_DATASET[dataset_id]["mfc"] == "none" for dataset_id in comparison_dataset_ids)
assert STYLE_BY_DATASET["harikane2025_spec"]["marker"] not in {"^", "v", "<", ">"}
assert STYLE_BY_DATASET["curtislake2026_jades_dr4_deep"]["marker"] not in {"^", "v", "<", ">", "1", "2", "3", "4"}

# All ten newly verified M_UV > -18 points, including discrepant measurements.
original_data = data.copy(deep=True)
original_figure_data = figure_data.copy(deep=True)
original_styles = {key: value.copy() for key, value in STYLE_BY_DATASET.items()}
search_data = pd.read_csv('UVLF_faint_end_search_20261016.txt', sep='\t', comment='#')
new_input = search_data.loc[search_data.already_in_current_figure.eq('no')].copy()
assert len(new_input) == 10
assert new_input.groupby('reference').size().to_dict() == {'A25': 2, 'L23': 4, 'P23': 4}
source_metadata = {
    'A25': ('adams2025_jof_faint', 'Adams et al.', 2025, 'arXiv:2502.10282',
            'Table 3, 0.2 arcsec catalogue', 'Poisson and cosmic variance in quadrature'),
    'L23': ('leung2023_ngdeep_faint', 'Leung et al.', 2023, '10.3847/2041-8213/acf365',
            'Table 2', '16th-84th posterior percentiles; Poisson and M_UV uncertainty'),
    'P23': ('perezgonzalez2023_midis_faint', 'Pérez-González et al.', 2023,
            '10.3847/2041-8213/acd9d0', 'Table 2',
            'Published logarithmic uncertainty converted to linear phi; cosmic variance included'),
}
new_rows = []
for record in new_input.itertuples(index=False):
    dataset, author, year, identifier, table, uncertainty = source_metadata[record.reference]
    width = 0.5 if record.reference == 'L23' and record.z_panel == 9 else 0.7 if record.reference == 'L23' else 1.0
    effective_z = 10.7 if record.reference == 'L23' and record.z_panel == 11 else (record.z_min + record.z_max) / 2
    row = {column: np.nan for column in REQUIRED_COLUMNS}
    row.update(dataset_id=dataset, reference=author, year=year, doi_or_arxiv=identifier,
               publication_status='published', source_table=table, redshift_panel=record.z_panel,
               z_effective=effective_z, z_min=record.z_min, z_max=record.z_max,
               M_UV=record.M_UV, M_UV_err_minus=width / 2, M_UV_err_plus=width / 2,
               phi=record.phi, phi_err_minus=record.err_minus, phi_err_plus=record.err_plus,
               limit_type='detection', uncertainty_definition=uncertainty, plot_flag=True,
               notes='New comparison only; not added to likelihood. Horizontal bars are magnitude-bin half-widths. '
                     'Raw published observational frame; no new dust or cosmology correction. '
                     'Leung/MIDIS share part of HUDF-Par2; broad redshift bins are shown at a representative panel.')
    new_rows.append(row)
new_data = pd.DataFrame(new_rows, columns=REQUIRED_COLUMNS)
data = pd.concat([data, new_data], ignore_index=True)
figure_data = pd.concat([figure_data, new_data], ignore_index=True)
validate_uvlf_data(data)
validate_uvlf_data(figure_data)
pd.testing.assert_frame_equal(data.iloc[:len(original_data)].reset_index(drop=True), original_data.reset_index(drop=True))
pd.testing.assert_frame_equal(figure_data.iloc[:len(original_figure_data)].reset_index(drop=True), original_figure_data.reset_index(drop=True))
assert len(data) == 178 and int(figure_data.plot_flag.sum()) == 173
assert new_data.groupby('redshift_panel').size().to_dict() == {9.: 5, 10.: 1, 11.: 3, 12.5: 1}
NEW_DATASET_STYLES = {
    'adams2025_jof_faint': ('o', '#CC79A7'),
    'leung2023_ngdeep_faint': ('s', '#56B4E9'),
    'perezgonzalez2023_midis_faint': ('+', '#E69F00'),
}
for dataset, (marker, color) in NEW_DATASET_STYLES.items():
    STYLE_BY_DATASET[dataset] = dict(marker=marker, color=color, mfc='none', mec=color,
        ms=marker_size_for_extent(marker), elinewidth=0.5, capsize=1.5, alpha=0.68, zorder=3)
comparison_dataset_ids += tuple(NEW_DATASET_STYLES)
plotted_dataset_ids += tuple(NEW_DATASET_STYLES)
REFERENCE_ABBREVIATIONS['Pérez-González'] = 'P.-Gonz.'
assert all(STYLE_BY_DATASET[key] == value for key, value in original_styles.items())
assert set(STYLE_BY_DATASET) == set(plotted_dataset_ids)
DATA_EXPORT_PATH = Path(f'{OUTPUT_STEM.name}_data.txt')
DATA_EXPORT_PATH.write_text(
    '# Original ledger preserved. Added ten verified faint-end comparison points, not likelihood inputs.\n'
    '# Actual redshift intervals and published uncertainty definitions are retained row by row.\n'
    '# New horizontal bars are magnitude-bin half-widths, not per-object measurement uncertainties.\n'
    '# Heterogeneous cosmologies/selection functions have not been homogenized.\n', encoding='utf-8')
figure_data.to_csv(DATA_EXPORT_PATH, sep='\t', index=False, mode='a', float_format='%.12g')
new_data.to_csv(f'{OUTPUT_STEM.name}_added_data.txt', sep='\t', index=False, float_format='%.12g')
caption_path = Path(f'{OUTPUT_STEM.name}_caption.tex')
caption = caption_path.read_text(encoding='utf-8').rstrip()
assert caption.endswith('}')
caption = caption.replace('Colored open symbols show independent recent measurements from',
                          'Colored open symbols show literature measurements from')
caption += '\n'
extra_caption = r""" Ten additional faint-end measurements are shown: two from Adams et al. (2025),
four from Leung et al. (2023), and four from Pérez-González et al. (2023)
\citep{Adams2025MediumBands,Leung2023NGDEEP,PerezGonzalez2023MIDIS}.
Their legend markers are a pink open circle, a blue open square, and a gold plus,
respectively; P.-Gonz. abbreviates Pérez-González.
All verified points with $M_{\rm UV}>-18$ from these three data selections are included,
not only those close to the model. New horizontal bars denote the reported magnitude-bin half-widths.
Adams bins $8.5<z<9.5$ and $9.5<z<11.5$ are displayed in panels $z=9$ and $z=10$;
Leung bins $8.5<z<9.5$ and $9.5<z<12$ are displayed at $z=9$ and $z=11$;
Pérez-González bins $8<z<10$, $10<z<11.5$, and $11.5<z<13$ are displayed at $z=9$, $11$, and $12.5$.
These representative panel assignments are for visual comparison, not selection-function-weighted model predictions.
The new data are not used to refit the model or appended to the likelihood.
NGDEEP and MIDIS partially overlap in HUDF-Par2, and the primary Donnan sample also uses NGDEEP/JADES;
these estimates must not all be treated as statistically independent.
Literature cosmologies and selection functions have not been homogenized in this preview."""
caption_path.write_text(caption.rstrip()[:-1] + extra_caption + '}\n', encoding='utf-8')
print('PASS: added exactly ten comparison points; all original observations and styles preserved.')

# User-requested display exclusions; no scientific values are deleted or refitted.
before_selection = figure_data.copy(deep=True)
REMOVED_DATASETS = {'chemerynska2026_glimpse', 'uncover2024', 'robertson2024_jof'}
removal_mask = figure_data.dataset_id.isin(REMOVED_DATASETS)
assert int((removal_mask & figure_data.plot_flag).sum()) == 23
removed_display_rows = figure_data.loc[removal_mask & figure_data.plot_flag].copy()
for frame in (data, figure_data):
    frame.loc[frame.dataset_id.isin(REMOVED_DATASETS), 'plot_flag'] = False
    whitler_mask = frame.dataset_id.str.startswith('whitler2026_jades_')
    frame.loc[whitler_mask, 'year'] = 2025
    frame.loc[whitler_mask, 'notes'] = frame.loc[whitler_mask, 'notes'].fillna('') + ' Display publication year corrected to 2025; ApJ 992, 63, doi:10.3847/1538-4357/adfddc.'
comparison_dataset_ids = tuple(k for k in comparison_dataset_ids if k not in REMOVED_DATASETS)
plotted_dataset_ids = tuple(k for k in plotted_dataset_ids if k not in REMOVED_DATASETS)
STYLE_BY_DATASET = {k: v for k, v in STYLE_BY_DATASET.items() if k not in REMOVED_DATASETS}
validate_uvlf_data(figure_data)
assert int(figure_data.plot_flag.sum()) == 150
assert not (figure_data.plot_flag & removal_mask).any()
pd.testing.assert_frame_equal(figure_data[['M_UV', 'phi', 'phi_err_minus', 'phi_err_plus']],
                              before_selection[['M_UV', 'phi', 'phi_err_minus', 'phi_err_plus']])
assert set(STYLE_BY_DATASET) == set(plotted_dataset_ids)
header = (
    '# Display revision: Whitler publication year corrected to 2025.\n'
    '# Chemerynska 2026, Atek 2024 UNCOVER and Robertson 2024 hidden at user request: 23 points.\n'
    '# All 178 ledger rows retained; plot_flag selects 150 displayed points.\n'
    '# The ten newly added faint-end points and both model curves are unchanged.\n'
)
DATA_EXPORT_PATH.write_text(header, encoding='utf-8')
figure_data.to_csv(DATA_EXPORT_PATH, sep='\t', index=False, mode='a', float_format='%.12g')
removed_display_rows.to_csv(f'{OUTPUT_STEM.name}_removed_data.txt', sep='\t', index=False, float_format='%.12g')
caption = caption_path.read_text(encoding='utf-8')
caption = caption.replace('Whitler+26', 'Whitler+25')
caption = caption.replace(' Atek et al.\n\\citep{Atek2024UNCOVER},', '')
caption = caption.replace(' Chemerynska et al.\n\\citep{Chemerynska2026GLIMPSE},', '')
caption = caption.replace('two-digit publication year; \\mbox{Chem.}, \\mbox{C.-Lake}, and\n\\mbox{R.-Ruiz} abbreviate Chemerynska, Curtis-Lake, and Rojas-Ruiz,',
                          'two-digit publication year; \\mbox{C.-Lake} and\n\\mbox{R.-Ruiz} abbreviate Curtis-Lake and Rojas-Ruiz,')
import re
caption = re.sub(r' Four Chemerynska et al\.\s+points centred.*?clipped\.', '', caption, flags=re.DOTALL)
caption_path.write_text(caption, encoding='utf-8')
print('PASS: 23 requested points hidden; 150 retained; Whitler labels corrected to 2025.')

caption = caption_path.read_text(encoding='utf-8')
caption = caption.replace('Robertson et al. \\citep{Robertson2024JADES}, ', '')
assert 'Robertson2024JADES' not in caption
caption_path.write_text(caption, encoding='utf-8')


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.text import Annotation, Text
from matplotlib.ticker import FixedLocator, FuncFormatter, LogLocator, NullFormatter
from matplotlib.transforms import offset_copy


def _row_asymmetric_error(row, minus_column, plus_column, *, allow_limit_extent=False):
    values = row[[minus_column, plus_column]].to_numpy(dtype=float)
    finite = np.isfinite(values)
    if finite.all():
        return values.reshape(2, 1)
    if allow_limit_extent:
        return np.where(finite, values, 0.0).reshape(2, 1)
    return None


def _row_display_yerr(row):
    limit_type = str(row["limit_type"])
    is_limit = limit_type in {"lower", "upper"}
    yerr = _row_asymmetric_error(
        row, "phi_err_minus", "phi_err_plus", allow_limit_extent=is_limit,
    )
    if yerr is None or not is_limit:
        return yerr
    yerr = yerr.copy()
    if limit_type == "lower":
        yerr[1, 0] = 0.0  # Hide the upward arm beneath the lower-limit arrow.
    else:
        yerr[0, 0] = 0.0  # Hide the downward arm beneath the upper-limit arrow.
    return yerr


def boundary_display_plan(frame):
    plotted = frame.loc[frame["plot_flag"]].copy()
    x_plus = plotted["M_UV_err_plus"].to_numpy(dtype=float)
    right_continuation = (
        np.isfinite(x_plus)
        & (plotted["M_UV"].to_numpy(dtype=float) <= X_LIMITS[1])
        & (plotted["M_UV"].to_numpy(dtype=float) + x_plus > X_LIMITS[1])
    )
    lower_boundary_proxy = (
        plotted["limit_type"].eq("lower").to_numpy()
        & (plotted["phi"].to_numpy(dtype=float) < Y_LIMITS[0])
    )
    log_lower_clip = (
        plotted["limit_type"].eq("detection").to_numpy()
        & np.isfinite(plotted["phi_err_minus"].to_numpy(dtype=float))
        & (plotted["phi"].to_numpy(dtype=float) - plotted["phi_err_minus"].to_numpy(dtype=float) <= 0)
    )
    boundary_rows = right_continuation | lower_boundary_proxy | log_lower_clip
    plan = plotted.loc[boundary_rows].copy()
    plan["display_kind"] = "log_lower_clip"
    plan.loc[right_continuation[boundary_rows], "display_kind"] = "right_continuation"
    plan.loc[lower_boundary_proxy[boundary_rows], "display_kind"] = "lower_boundary_proxy"
    plan["visible_xerr_plus"] = plan["M_UV_err_plus"]
    plan.loc[plan["display_kind"].eq("right_continuation"), "visible_xerr_plus"] = 0.0
    plan["display_y"] = plan["phi"]
    plan.loc[plan["display_kind"].eq("lower_boundary_proxy"), "display_y"] = Y_LIMITS[0]
    return plan


def semantic_marker_bbox_records(rendered_rows, renderer):
    records = []
    for record in rendered_rows.itertuples():
        central_line = record.artist.lines[0]
        records.append({
            "axis": central_line.axes,
            "kind": "lower_proxy_caret" if record.display_kind == "lower_boundary_proxy" else "central_marker",
            "dataset_id": record.dataset_id,
            "row_identity": record.row_identity,
            "bbox": central_line.get_window_extent(renderer).frozen(),
        })
    return records


def arrow_patch_display_bbox(annotation):
    return annotation.arrow_patch.get_path().transformed(annotation.arrow_patch.get_transform()).get_extents()


def bbox_edge_clearance_mm(first, second, figure):
    dx = max(second.x0 - first.x1, first.x0 - second.x1, 0.0)
    dy = max(second.y0 - first.y1, first.y0 - second.y1, 0.0)
    return float(np.hypot(dx, dy) * 25.4 / figure.dpi)


def _padded_bboxes_are_disjoint(first, second, clearance_pixels):
    half_padding = clearance_pixels / 2.0
    return not first.padded(half_padding).overlaps(second.padded(half_padding))


def _arrow_candidate_is_clear(annotation, arrow_bbox, rendered_rows, renderer, clearance_pixels):
    axis = annotation.axes
    inner_axes_bbox = axis.bbox.padded(-clearance_pixels)
    if not (
        arrow_bbox.x0 >= inner_axes_bbox.x0 and arrow_bbox.y0 >= inner_axes_bbox.y0
        and arrow_bbox.x1 <= inner_axes_bbox.x1 and arrow_bbox.y1 <= inner_axes_bbox.y1
    ):
        return False
    semantic_records = semantic_marker_bbox_records(rendered_rows, renderer)
    if any(
        record["axis"] is axis
        and not _padded_bboxes_are_disjoint(arrow_bbox, record["bbox"], clearance_pixels)
        for record in semantic_records
    ):
        return False
    visible_text_bboxes = [
        text.get_window_extent(renderer).frozen() for text in axis.findobj(Text)
        if text is not annotation and text.get_visible() and text.get_text().strip()
    ]
    return all(_padded_bboxes_are_disjoint(arrow_bbox, bbox, clearance_pixels) for bbox in visible_text_bboxes)


def place_boundary_arrows_with_clearance(figure, panel_axes, arrow_annotations, rendered_rows):
    figure.canvas.draw()
    panel_by_axis = {axis: panel for panel, axis in panel_axes.items()}
    clearance_pixels = ARROW_MARKER_CLEARANCE_MM * figure.dpi / 25.4
    placements = []
    for annotation in arrow_annotations:
        is_right_arrow = np.isclose(annotation.xy[1], annotation.xyann[1])
        candidates = RIGHT_ARROW_OFFSET_CANDIDATES_PT if is_right_arrow else LOWER_ARROW_OFFSET_CANDIDATES_PT
        for x_offset_pt, y_offset_pt in candidates:
            offset_transform = offset_copy(
                annotation.axes.transData, fig=figure, x=x_offset_pt, y=y_offset_pt, units="points",
            )
            annotation.xycoords = offset_transform
            annotation.set_anncoords(offset_transform)
            figure.canvas.draw()
            renderer = figure.canvas.get_renderer()
            arrow_bbox = arrow_patch_display_bbox(annotation)
            if _arrow_candidate_is_clear(annotation, arrow_bbox, rendered_rows, renderer, clearance_pixels):
                same_panel_records = [
                    record for record in semantic_marker_bbox_records(rendered_rows, renderer)
                    if record["axis"] is annotation.axes
                ]
                placements.append({
                    "annotation": annotation, "redshift_panel": panel_by_axis[annotation.axes],
                    "x_offset_pt": x_offset_pt, "y_offset_pt": y_offset_pt,
                    "minimum_marker_clearance_mm": min(
                        bbox_edge_clearance_mm(arrow_bbox, record["bbox"], figure) for record in same_panel_records
                    ),
                })
                break
        else:
            raise AssertionError(f"No collision-free display-space placement for panel {panel_by_axis[annotation.axes]}")
    figure.canvas.draw()
    return placements


def _draw_right_continuation(ax, row, style):
    return ax.annotate(
        "", xy=(X_LIMITS[1], float(row["phi"])),
        xytext=(X_LIMITS[1] - 0.50, float(row["phi"])),
        arrowprops=dict(arrowstyle="-|>", mutation_scale=BOUNDARY_ARROW_MUTATION_SCALE_PT, color=style["color"], lw=max(0.65, style["elinewidth"])),
        annotation_clip=False, zorder=style["zorder"] + 0.2,
    )


def _draw_lower_boundary_proxy(ax, row, style, xerr):
    proxy_style = {key: value for key, value in style.items() if key not in {"marker", "color", "mfc", "mec", "ms"}}
    proxy = ax.errorbar(
        [float(row["M_UV"])], [Y_LIMITS[0]], xerr=xerr, fmt="^", linestyle="none",
        color=style["color"], markerfacecolor="white", markeredgecolor=style["color"],
        markersize=marker_size_for_extent("^", SEMANTIC_MARKER_EXTENT_PT), label="_nolegend_", clip_on=False, **proxy_style,
    )
    arrow = ax.annotate(
        "", xy=(float(row["M_UV"]), Y_LIMITS[0] * 4.0),
        xytext=(float(row["M_UV"]), Y_LIMITS[0]),
        arrowprops=dict(arrowstyle="-|>", mutation_scale=BOUNDARY_ARROW_MUTATION_SCALE_PT, color=style["color"], lw=max(0.65, style["elinewidth"])),
        annotation_clip=False, zorder=style["zorder"] + 0.2,
    )
    assert proxy.lines[0].get_marker() == "^"
    assert arrow.xy[1] > arrow.xyann[1]
    return [proxy, arrow]


def _draw_one_sided_limit_arrow(ax, row, style):
    limit_type = str(row["limit_type"])
    assert limit_type in {"lower", "upper"}
    direction = 1.0 if limit_type == "lower" else -1.0
    annotation = ax.annotate(
        "", xy=(float(row["M_UV"]), float(row["phi"])),
        xytext=(0.0, direction * LIMIT_ARROW_OFFSET_PT), textcoords="offset points",
        arrowprops=dict(
            arrowstyle="<-", color=style["color"], fill=False,
            lw=LIMIT_ARROW_LINEWIDTH_PT, mutation_scale=LIMIT_ARROW_MUTATION_SCALE_PT,
            shrinkA=0.0, shrinkB=0.0,
        ),
        annotation_clip=True, zorder=style["zorder"] - 0.1,
    )
    annotation.set_gid("one-sided-limit-arrow")
    annotation.arrow_patch.set_fill(False)
    annotation.arrow_patch.set_clip_path(ax.patch)
    return annotation


def plot_dataset(ax, subset, style, boundary_display, rendered_rows):
    artists = []
    errorbar_style = {key: value for key, value in style.items() if key != "marker"}
    for row_index, row in subset.sort_values("M_UV").iterrows():
        row_frame = row.to_frame().T
        uplims = row_frame["limit_type"].eq("upper").to_numpy()
        lolims = row_frame["limit_type"].eq("lower").to_numpy()
        xerr = _row_asymmetric_error(row, "M_UV_err_minus", "M_UV_err_plus")
        yerr = _row_display_yerr(row)
        display_kind = boundary_display.loc[row_index, "display_kind"] if row_index in boundary_display.index else "normal"
        if display_kind == "lower_boundary_proxy":
            raise AssertionError("Out-of-range central values must be excluded before rendering")
        else:
            if display_kind == "right_continuation":
                xerr = xerr.copy()
                xerr[1, 0] = float(boundary_display.loc[row_index, "visible_xerr_plus"])
            marker_artist = ax.errorbar(
                [float(row["M_UV"])], [float(row["phi"])],
                xerr=xerr, yerr=yerr,
                fmt=style["marker"], linestyle="none", label="_nolegend_",
                clip_on=display_kind == "normal" or display_kind == "log_lower_clip",
                **errorbar_style,
            )
            limit_annotation = _draw_one_sided_limit_arrow(ax, row, style) if bool(uplims[0] or lolims[0]) else None
            artists.append(marker_artist)
        row_identity = (
            str(row["dataset_id"]), float(row["redshift_panel"]),
            float(row["M_UV"]), float(row["phi"]), str(row["limit_type"]),
        )
        rendered_rows.append({
            "row_identity": row_identity, "row_index": int(row_index),
            "redshift_panel": float(row["redshift_panel"]),
            "dataset_id": str(row["dataset_id"]), "limit_type": str(row["limit_type"]),
            "display_kind": display_kind,
            "limit_direction": "upward" if bool(lolims[0]) else "downward" if bool(uplims[0]) else "none",
            "artist": marker_artist, "limit_annotation": limit_annotation,
        })
    return artists


def place_one_sided_limit_arrows(figure, rendered_rows):
    placements = []
    for record in rendered_rows.itertuples():
        annotation = record.limit_annotation
        if annotation is None:
            continue
        direction = 1.0 if record.limit_type == "lower" else -1.0
        x_value, y_value = record.row_identity[2], record.row_identity[3]
        annotation.xy = (x_value, y_value)
        annotation.xyann = (LIMIT_ARROW_X_OFFSET_PT, direction * LIMIT_ARROW_OFFSET_PT)
        annotation.xycoords = "data"
        annotation.set_anncoords("offset points")
        placements.append({
            "row_identity": record.row_identity, "x_offset_pt": LIMIT_ARROW_X_OFFSET_PT,
            "y_offset_pt": direction * LIMIT_ARROW_OFFSET_PT,
        })
    figure.canvas.draw()
    return placements


def make_uvlf_figure(frame, theory_frame):
    figure_rc = {
        "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "font.size": 8.0,
        "axes.labelsize": 8.0, "axes.titlesize": 8.0,
        "xtick.labelsize": 8.0, "ytick.labelsize": 8.0,
        "axes.linewidth": 0.6, "xtick.major.width": 0.6,
        "ytick.major.width": 0.6, "xtick.minor.width": 0.5,
        "ytick.minor.width": 0.5, "xtick.direction": "in",
        "ytick.direction": "in", "figure.facecolor": "white",
        "axes.facecolor": "white", "savefig.facecolor": "white",
        "axes.spines.top": False, "axes.spines.right": False,
    }
    with mpl.rc_context(figure_rc):
        fig, axes = plt.subplots(
            2, 4, figsize=(7.0, 4.25), sharex=True, sharey=True,
            layout="constrained",
        )
        flat_axes = axes.ravel()
        panel_axes = dict(zip(PANELS, [flat_axes[index] for index in PANEL_SLOT_INDICES]))
        legend_ax = flat_axes[LEGEND_SLOT_INDEX]
        theory_line_artists = {}

        plotted = frame.loc[frame["plot_flag"]]
        boundary_display = boundary_display_plan(frame)
        rendered_rows = []
        for panel_index, (redshift, ax) in enumerate(panel_axes.items()):
            panel_theory = theory_frame.loc[theory_frame["redshift"].astype(float).eq(redshift)]
            if set(panel_theory["model_id"]) != set(DISPLAY_MODEL_IDS):
                raise ValueError(f"Missing theory family for z={redshift}")
            for model_id, model_label, line_style in zip(
                DISPLAY_MODEL_IDS, DISPLAY_MODEL_LABELS, DISPLAY_MODEL_LINESTYLES,
            ):
                panel_model = panel_theory.loc[panel_theory["model_id"].eq(model_id)].sort_values("M_UV")
                theory_line, = ax.plot(
                    panel_model["M_UV"], panel_model["phi_observed"],
                    color=THEORY_LINE_COLOR, linestyle=line_style,
                    linewidth=THEORY_LINEWIDTH_PT, zorder=THEORY_LINE_ZORDER,
                    label="_nolegend_", clip_on=True,
                )
                theory_line_artists[(redshift, model_id)] = theory_line
            panel = plotted.loc[plotted["redshift_panel"].astype(float).eq(redshift)]
            dataset_order = sorted(
                panel["dataset_id"].unique(),
                key=lambda dataset_id: (dataset_id in PRIMARY_STYLES, dataset_id),
            )
            for dataset_id in dataset_order:
                plot_dataset(ax, panel.loc[panel["dataset_id"].eq(dataset_id)], STYLE_BY_DATASET[dataset_id], boundary_display, rendered_rows)

            ax.set(xlim=X_LIMITS, ylim=Y_LIMITS, yscale="log")
            ax.xaxis.set_major_locator(FixedLocator([-24, -22, -20, -18, -16]))
            ax.yaxis.set_major_locator(LogLocator(base=10, numticks=8))
            ax.yaxis.set_major_formatter(FuncFormatter(lambda value, position: f"1e{int(np.log10(value))}"))
            ax.yaxis.set_minor_locator(LogLocator(base=10, subs=(2, 5), numticks=16))
            ax.yaxis.set_minor_formatter(NullFormatter())
            ax.tick_params(which="both", top=False, right=False)
            ax.tick_params(which="major", length=3.0)
            ax.tick_params(which="minor", length=1.7)
            redshift_label = "12.5" if redshift == 12.5 else str(int(redshift))
            ax.text(
                0.05, 0.95, f"({chr(97 + panel_index)})  $z={redshift_label}$",
                transform=ax.transAxes, ha="left", va="top", fontsize=8.0, fontweight="bold",
                zorder=20,
            )

        legend_ax.axis("off")
        legend_order = list(PRIMARY_STYLES) + list(comparison_dataset_ids)
        handles = []
        for dataset_id in legend_order:
            style = STYLE_BY_DATASET[dataset_id]
            handles.append(Line2D(
                [], [], linestyle="none", marker=style["marker"],
                color=style["color"], markerfacecolor=style["mfc"],
                markeredgecolor=style["mec"], markersize=style["ms"],
                markeredgewidth=max(0.55, style["elinewidth"]),
                alpha=style["alpha"], label=compact_reference_label(dataset_id, frame),
            ))
        dataset_legend = legend_ax.legend(
            handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.99), ncol=2, frameon=False,
            fontsize=7.5, handletextpad=0.3, handlelength=0.9,
            columnspacing=0.55, labelspacing=0.28, borderaxespad=0.0,
        )
        legend_ax.add_artist(dataset_legend)
        theory_handles = [
            Line2D(
                [], [], color=THEORY_LINE_COLOR, linestyle=line_style,
                linewidth=THEORY_LINEWIDTH_PT, label={"Fixed SF": "Fixed cosmology + astrophysics", "Free SF": "Fixed cosmology"}.get(label, label),
            )
            for label, line_style in zip(DISPLAY_MODEL_LABELS, DISPLAY_MODEL_LINESTYLES)
        ]
        theory_legend = legend_ax.legend(
            handles=theory_handles, loc="lower center", bbox_to_anchor=(0.5, THEORY_LEGEND_Y_AXES),
            ncol=1, frameon=False, fontsize=8.0, handletextpad=0.3,
            handlelength=1.5, labelspacing=0.28, borderaxespad=0.0,
        )
        fig.supxlabel(r"$M_{\mathrm{UV}}\;[\mathrm{mag}]$", fontsize=8.0, math_fontfamily="stix")
        fig.supylabel(r"$\Phi_{\mathrm{UV}}\;[\mathrm{Mpc}^{-3}\,\mathrm{mag}^{-1}]$", fontsize=8.0, math_fontfamily="stix")

    fig.theory_line_artists = theory_line_artists
    fig.dataset_legend = dataset_legend
    fig.theory_legend = theory_legend
    fig.panel_slot_indices = PANEL_SLOT_INDICES
    fig.legend_slot_index = LEGEND_SLOT_INDEX
    fig.boundary_display = boundary_display
    fig.limit_arrow_annotations = [
        child for ax in panel_axes.values() for child in ax.get_children()
        if isinstance(child, Annotation) and child.arrow_patch is not None
        and child.get_gid() == "one-sided-limit-arrow"
    ]
    fig.boundary_arrow_annotations = [
        child for ax in panel_axes.values() for child in ax.get_children()
        if isinstance(child, Annotation) and child.arrow_patch is not None
        and child.get_gid() != "one-sided-limit-arrow"
    ]
    fig.observational_artist_rows = pd.DataFrame(rendered_rows)
    fig.limit_arrow_placements = place_one_sided_limit_arrows(
        fig, fig.observational_artist_rows,
    )
    fig.boundary_arrow_placements = place_boundary_arrows_with_clearance(
        fig, panel_axes, fig.boundary_arrow_annotations, fig.observational_artist_rows,
    )
    # Preserve the previous panel geometry when removing a legend entry.
    fig.set_layout_engine("none")
    original_positions = [[0.08394587301587302, 0.5594460130718955, 0.1751186011904762, 0.43074928104575166], [0.29424057539682535, 0.5594460130718955, 0.1751186011904762, 0.43074928104575166], [0.5045352777777777, 0.5594460130718955, 0.1751186011904762, 0.43074928104575166], [0.08394587301587302, 0.0949696732026144, 0.1751186011904762, 0.43074928104575166], [0.29424057539682535, 0.0949696732026144, 0.1751186011904762, 0.43074928104575166], [0.5045352777777777, 0.0949696732026144, 0.1751186011904762, 0.43074928104575166], [0.7584417619047619, 0.0949696732026144, 0.1751186011904762, 0.43074928104575166]]
    for ax, position in zip(panel_axes.values(), original_positions):
        ax.set_position(position)
    legend_ax.set_position([original_positions[-1][0], original_positions[0][1],
                            original_positions[-1][2], original_positions[0][3]])
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    dataset_bbox = fig.dataset_legend.get_window_extent(renderer=renderer)
    theory_bbox = fig.theory_legend.get_window_extent(renderer=renderer)
    fig.legend_vertical_gap_pt = (dataset_bbox.y0 - theory_bbox.y1) * 72.0 / fig.dpi
    if fig.legend_vertical_gap_pt < LEGEND_MIN_VERTICAL_GAP_PT:
        raise AssertionError(
            f"Legend gap {fig.legend_vertical_gap_pt:.2f} pt is below "
            f"the {LEGEND_MIN_VERTICAL_GAP_PT:.2f} pt minimum"
        )
    fig.set_layout_engine("none")
    for text, label in zip(theory_legend.get_texts(), DISPLAY_MODEL_LABELS):
        text.set_text(label)
    return fig, panel_axes, legend_ax


In [ ]:
fig, panel_axes, legend_ax = make_uvlf_figure(figure_data, theory_curve_data)
assert tuple(panel_axes) == PANELS
assert len(fig.axes) == 8
assert all(ax.get_yscale() == "log" for ax in panel_axes.values())
assert all(tuple(ax.get_xlim()) == X_LIMITS for ax in panel_axes.values())
assert all(np.allclose(ax.get_ylim(), Y_LIMITS) for ax in panel_axes.values())
assert not legend_ax.axison
assert sum(len(ax.lines) for ax in panel_axes.values()) > 0
assert all(len(ax.collections) + len(ax.lines) > 0 for ax in panel_axes.values())
assert set(figure_data.loc[figure_data.plot_flag, "dataset_id"]) == set(STYLE_BY_DATASET)
rendered_rows = fig.observational_artist_rows
assert len(rendered_rows) == int(figure_data.loc[figure_data.plot_flag].shape[0]) == 150
assert rendered_rows["row_identity"].is_unique
assert set(rendered_rows["row_index"]) == set(figure_data.index[figure_data.plot_flag])
rendered_summary = (
    rendered_rows.groupby(["redshift_panel", "dataset_id"], observed=True)
    .size().rename("n_points").reset_index()
    .sort_values(["redshift_panel", "dataset_id"]).reset_index(drop=True)
)
pd.testing.assert_frame_equal(rendered_summary, summarize_uvlf_data(figure_data).reset_index(drop=True), check_dtype=False)
central_marker_records = []
comparison_marker_face_alphas = []
for record in rendered_rows.itertuples():
    central_line = record.artist.lines[0]
    width_pt, height_pt = transformed_marker_extent_points(central_line.get_marker(), central_line.get_markersize())
    central_marker_records.append({"dataset_id": record.dataset_id, "extent_pt": min(width_pt, height_pt), "row_identity": record.row_identity})
    if record.dataset_id not in PRIMARY_STYLES:
        comparison_marker_face_alphas.append(mpl.colors.to_rgba(central_line.get_markerfacecolor())[3])
central_marker_extents = pd.DataFrame(central_marker_records)
assert len(central_marker_extents) == len(rendered_rows)
assert np.allclose(central_marker_extents["extent_pt"], CENTRAL_MARKER_EXTENT_PT, rtol=0.0, atol=CENTRAL_MARKER_EQUALITY_TOLERANCE_PT)
minimum_central_marker_extent_pt = float(central_marker_extents["extent_pt"].min())
primary_central_extents = central_marker_extents.loc[central_marker_extents["dataset_id"].isin(PRIMARY_STYLES), "extent_pt"]
comparison_central_extents = central_marker_extents.loc[~central_marker_extents["dataset_id"].isin(PRIMARY_STYLES), "extent_pt"]
assert len(primary_central_extents) == 55 and not comparison_central_extents.empty
primary_central_extent_pt = float(primary_central_extents.mean())
comparison_central_extent_pt = float(comparison_central_extents.mean())
assert np.isclose(primary_central_extent_pt, comparison_central_extent_pt, rtol=0.0, atol=CENTRAL_MARKER_EQUALITY_TOLERANCE_PT)
assert np.isclose(primary_central_extent_pt, CENTRAL_MARKER_EXTENT_PT, rtol=0.0, atol=CENTRAL_MARKER_EQUALITY_TOLERANCE_PT)
assert np.allclose(comparison_marker_face_alphas, 0.0)
for redshift in PANELS:
    panel_comparisons = sorted(
        set(figure_data.loc[figure_data.plot_flag & figure_data.redshift_panel.astype(float).eq(redshift), "dataset_id"])
        - set(PRIMARY_STYLES)
    )
    panel_markers = [STYLE_BY_DATASET[dataset_id]["marker"] for dataset_id in panel_comparisons]
    assert len(panel_markers) == len(set(panel_markers)), (redshift, panel_comparisons, panel_markers)
assert all(
    line.get_linestyle().lower() == "none"
    for ax in panel_axes.values() for line in ax.lines
    if line not in fig.theory_line_artists.values()
)
assert all(not ax.spines["top"].get_visible() and not ax.spines["right"].get_visible() for ax in panel_axes.values())
assert all(not gridline.get_visible() for ax in panel_axes.values() for gridline in (*ax.get_xgridlines(), *ax.get_ygridlines()))
assert legend_ax.get_legend() is not None and not legend_ax.get_legend().get_frame_on()
expected_theory_keys = {(redshift, model_id) for redshift in PANELS for model_id in DISPLAY_MODEL_IDS}
assert set(fig.theory_line_artists) == expected_theory_keys
assert len(fig.theory_line_artists) == 14
for (redshift, model_id), theory_line in fig.theory_line_artists.items():
    panel_curve = theory_curve_data.loc[
        theory_curve_data["redshift"].eq(redshift)
        & theory_curve_data["model_id"].eq(model_id)
    ].sort_values("M_UV")
    assert np.allclose(theory_line.get_xdata(), panel_curve["M_UV"])
    assert np.allclose(theory_line.get_ydata(), panel_curve["phi_observed"])
    assert mpl.colors.to_rgba(theory_line.get_color()) == mpl.colors.to_rgba("black")
    expected_style = DISPLAY_MODEL_LINESTYLES[DISPLAY_MODEL_IDS.index(model_id)]
    assert theory_line.get_linestyle() == expected_style
    assert np.isclose(theory_line.get_linewidth(), THEORY_LINEWIDTH_PT)
    panel_observations = rendered_rows.loc[rendered_rows["redshift_panel"].eq(redshift)]
    assert theory_line.get_zorder() < min(record.artist.lines[0].get_zorder() for record in panel_observations.itertuples())
dataset_legend_labels = [text.get_text() for text in fig.dataset_legend.get_texts()]
theory_legend_labels = [text.get_text() for text in fig.theory_legend.get_texts()]
assert len(dataset_legend_labels) == len(STYLE_BY_DATASET)
assert not set(THEORY_MODEL_LABELS).intersection(dataset_legend_labels)
assert theory_legend_labels == list(DISPLAY_MODEL_LABELS)
assert fig.panel_slot_indices == (0, 1, 2, 4, 5, 6, 7)
assert fig.legend_slot_index == 3
print("Panel layout validated: top 6,7,8,legend; bottom 9,10,11,12.5")
print("Theory artists validated: 14 black curves with solid and dotted styles.")
assert fig.legend_vertical_gap_pt >= LEGEND_MIN_VERTICAL_GAP_PT
print(f"Legend vertical gap validated: {fig.legend_vertical_gap_pt:.2f} pt")
# Boundary-display contract for the implemented continuation and clipping conventions.
boundary_display = boundary_display_plan(figure_data)
right_boundary = boundary_display.loc[boundary_display["display_kind"].eq("right_continuation")]
lower_boundary = boundary_display.loc[boundary_display["display_kind"].eq("lower_boundary_proxy")]
assert len(right_boundary) == 0
assert set(right_boundary["redshift_panel"]) == set()
assert np.allclose(right_boundary["M_UV"], X_LIMITS[1])
assert np.allclose(right_boundary["visible_xerr_plus"], 0.0)
assert lower_boundary.empty
lower_rendered = rendered_rows.loc[rendered_rows["limit_type"].eq("lower")]
assert not lower_rendered.empty and lower_rendered["limit_direction"].eq("upward").all()
assert lower_rendered.loc[lower_rendered["display_kind"].eq("lower_boundary_proxy")].empty
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
limit_arrow_annotations = fig.limit_arrow_annotations
visible_limit_rows = rendered_rows.loc[rendered_rows["limit_type"].isin({"lower", "upper"})]
assert len(limit_arrow_annotations) == len(visible_limit_rows) == 11
assert len(fig.limit_arrow_placements) == len(limit_arrow_annotations)
filled_limit_caret_count = sum(
    cap.get_marker() in {10, 11}
    for record in rendered_rows.itertuples() for cap in record.artist.lines[1]
)
assert filled_limit_caret_count == 0
assert all(not annotation.arrow_patch.get_fill() for annotation in limit_arrow_annotations)
assert all(np.isclose(annotation.arrow_patch.get_linewidth(), LIMIT_ARROW_LINEWIDTH_PT) for annotation in limit_arrow_annotations)
limit_placement_by_identity = {placement["row_identity"]: placement for placement in fig.limit_arrow_placements}
for record in visible_limit_rows.itertuples():
    assert record.limit_annotation is not None
    assert record.limit_annotation.get_gid() == "one-sided-limit-arrow"
    expected_sign = 1.0 if record.limit_type == "lower" else -1.0
    assert np.isclose(limit_placement_by_identity[record.row_identity]["x_offset_pt"], 0.0)
    assert np.sign(limit_placement_by_identity[record.row_identity]["y_offset_pt"]) == expected_sign
boundary_arrow_annotations = fig.boundary_arrow_annotations
assert len(boundary_arrow_annotations) == 0
assert len(fig.boundary_arrow_placements) == len(boundary_arrow_annotations)
limit_arrow_boxes = [arrow_patch_display_bbox(annotation) for annotation in limit_arrow_annotations]
for record, arrow_box in zip(visible_limit_rows.itertuples(), limit_arrow_boxes):
    marker_x_px = record.limit_annotation.axes.transData.transform((record.row_identity[2], record.row_identity[3]))[0]
    arrow_path = record.limit_annotation.arrow_patch.get_path().transformed(record.limit_annotation.arrow_patch.get_transform())
    arrow_x_px = float(np.median(arrow_path.vertices[:, 0]))
    assert abs(arrow_x_px - marker_x_px) * 72.0 / fig.dpi <= 0.2
    assert arrow_box.x0 <= marker_x_px <= arrow_box.x1
assert int(boundary_display["display_kind"].eq("log_lower_clip").sum()) == 4
normally_visible = figure_data.loc[figure_data.plot_flag & ~figure_data.index.isin(boundary_display.index)]
assert len(data) == 178
assert len(display_excluded) == 1
assert len(boundary_display) + len(normally_visible) == int(figure_data.loc[figure_data.plot_flag].shape[0]) == 150
assert normally_visible["M_UV"].between(*X_LIMITS).all()
assert normally_visible["phi"].between(*Y_LIMITS).all()
assert (normally_visible["M_UV"] + normally_visible["M_UV_err_plus"].fillna(0) <= X_LIMITS[1]).all()
assert (normally_visible["M_UV"] - normally_visible["M_UV_err_minus"].fillna(0) >= X_LIMITS[0]).all()
assert (normally_visible["phi"] + normally_visible["phi_err_plus"].fillna(0) <= Y_LIMITS[1]).all()
assert (normally_visible["phi"] - normally_visible["phi_err_minus"].fillna(0) > 0).all()
harikane_z7 = data.loc[data.plot_flag & data.dataset_id.eq("harikane2025_spec") & data.redshift_panel.astype(float).eq(7.0)].sort_values("M_UV")
expected_harikane_z7 = {
    -23.7: ("lower", 2.3e-9, 1.9e-9, 5.3e-9),
    -23.2: ("detection", 1.6e-7, 1.3e-7, 3.7e-7),
    -22.7: ("detection", 4.8e-7, 2.6e-7, 4.6e-7),
    -22.2: ("lower", 1.1e-6, 0.4e-6, 0.4e-6),
    -21.7: ("lower", 2.4e-5, 1.7e-5, 3.3e-5),
    -21.2: ("lower", 1.2e-5, 1.1e-5, 2.8e-5),
}
assert len(harikane_z7) == len(expected_harikane_z7)
for row in harikane_z7.itertuples():
    expected = expected_harikane_z7[row.M_UV]
    assert row.limit_type == expected[0]
    assert np.allclose([row.phi, row.phi_err_minus, row.phi_err_plus], expected[1:])
print(
    f"Figure assertions passed: {len(rendered_rows)} rows; central markers "
    f"primary={primary_central_extent_pt * 25.4 / 72:.3f} mm, "
    f"comparison={comparison_central_extent_pt * 25.4 / 72:.3f} mm; "
    f"display-excluded={len(display_excluded)}; limit arrows={len(limit_arrow_annotations)}; "
    f"filled limit carets={filled_limit_caret_count}; boundary arrows={len(boundary_arrow_annotations)}; "
    f"error-bar capsize={PRIMARY_STYLES['bouwens2021_primary']['capsize']:.1f} pt; "
    f"line widths primary={PRIMARY_STYLES['bouwens2021_primary']['elinewidth']:.2f} pt, "
    f"comparison={STYLE_BY_DATASET[comparison_dataset_ids[0]]['elinewidth']:.2f} pt."
)

assert rendered_rows.loc[rendered_rows.dataset_id.isin(NEW_DATASET_STYLES)].groupby('dataset_id').size().to_dict() == {
    'adams2025_jof_faint': 2, 'leung2023_ngdeep_faint': 4, 'perezgonzalez2023_midis_faint': 4}
renderer = fig.canvas.get_renderer()
for legend in (fig.dataset_legend, fig.theory_legend):
    bounds = legend.get_window_extent(renderer)
    assert bounds.x0 >= 0 and bounds.x1 <= fig.bbox.width, (bounds.bounds, fig.bbox.bounds)
    assert bounds.y0 >= 0 and bounds.y1 <= fig.bbox.height
previous_theory = pd.read_csv('UVLF_full_WithPBH_solid_theory.txt', sep='\t', comment='#')
np.testing.assert_array_equal(theory_curve_data[['M_UV','phi_observed']].to_numpy(),
                              previous_theory[['M_UV','phi_observed']].to_numpy())
print('PASS: 150 points total; ten added exactly once; both model curves unchanged.')

assert not set(rendered_rows.dataset_id).intersection(REMOVED_DATASETS)
assert not any('Chem.' in s or 'Atek+' in s or 'Whitler+26' in s for s in dataset_legend_labels)
assert {'Whitler+25 115', 'Whitler+25 150'}.issubset(dataset_legend_labels)
print('PASS: removed datasets absent from all panels and legend; both Whitler labels verified.')

assert not any('Robertson' in label for label in dataset_legend_labels)


In [ ]:
import re


def canonicalize_svg_clip_path_ids(svg_text):
    clip_path_pattern = re.compile(r'(<clipPath\b[^>]*\bid=")(.*?)(")')
    clip_path_ids = [match.group(2) for match in clip_path_pattern.finditer(svg_text)]
    assert len(clip_path_ids) == len(set(clip_path_ids)), clip_path_ids
    all_ids = set(re.findall(r'\bid="([^"]+)"', svg_text))
    reserved_ids = all_ids - set(clip_path_ids)
    assigned_ids = set()
    clip_id_mapping = {}
    next_ordinal = 1
    for original_id in clip_path_ids:
        while True:
            candidate_id = f"clip{next_ordinal:04d}"
            next_ordinal += 1
            if candidate_id not in reserved_ids and candidate_id not in assigned_ids:
                break
        clip_id_mapping[original_id] = candidate_id
        assigned_ids.add(candidate_id)

    def replace_clip_path_id(match):
        return f"{match.group(1)}{clip_id_mapping[match.group(2)]}{match.group(3)}"

    canonical_svg = clip_path_pattern.sub(replace_clip_path_id, svg_text)
    canonical_svg = re.sub(
        r'url\(#([^\)]+)\)',
        lambda match: f"url(#{clip_id_mapping.get(match.group(1), match.group(1))})",
        canonical_svg,
    )
    canonical_ids = [match.group(2) for match in clip_path_pattern.finditer(canonical_svg)]
    assert canonical_ids == list(clip_id_mapping.values())
    assert assigned_ids.isdisjoint(reserved_ids)
    return canonical_svg, clip_id_mapping


collision_probe = '<svg><g id="clip0001"/><clipPath id="process-id"/><g clip-path="url(#process-id)"/></svg>'
canonical_probe, collision_probe_mapping = canonicalize_svg_clip_path_ids(collision_probe)
assert collision_probe_mapping == {"process-id": "clip0002"}
assert 'id="clip0001"' in canonical_probe and 'url(#clip0002)' in canonical_probe


export_paths = {
    "pdf": OUTPUT_STEM.with_suffix(".pdf"),
    "svg": OUTPUT_STEM.with_suffix(".svg"),
    "png": OUTPUT_STEM.with_suffix(".png"),
}
fig.savefig(
    export_paths["pdf"], format="pdf", facecolor="white", bbox_inches=None,
    metadata={"CreationDate": ARTIFACT_TIMESTAMP, "ModDate": ARTIFACT_TIMESTAMP},
)
fig.savefig(
    export_paths["svg"], format="svg", facecolor="white", bbox_inches=None,
    metadata={"Date": ARTIFACT_TIMESTAMP.date().isoformat()},
)
# Matplotlib emits trailing spaces inside SVG path serialization; normalize them so git's whitespace check and byte-level replay are stable.
svg_text = export_paths["svg"].read_text(encoding="utf-8")
svg_text, svg_clip_id_mapping = canonicalize_svg_clip_path_ids(svg_text)
assert list(svg_clip_id_mapping.values()) == [f"clip{index:04d}" for index in range(1, len(panel_axes) + 1)]
export_paths["svg"].write_text(
    "\n".join(line.rstrip() for line in svg_text.splitlines()) + "\n",
    encoding="utf-8",
)
fig.savefig(export_paths["png"], format="png", dpi=600, facecolor="white", bbox_inches=None)


In [ ]:
import io
import math

from fontTools.ttLib import TTFont
from PIL import Image
from pypdf import PdfReader

for path in export_paths.values():
    assert path.exists() and path.stat().st_size > 0, path

reader = PdfReader(export_paths["pdf"])
assert len(reader.pages) == 1

def _multiply_pdf_matrices(left, right):
    a1, b1, c1, d1, e1, f1 = left
    a2, b2, c2, d2, e2, f2 = right
    return [
        a1 * a2 + b1 * c2, a1 * b2 + b1 * d2,
        c1 * a2 + d1 * c2, c1 * b2 + d1 * d2,
        e1 * a2 + f1 * c2 + e2, e1 * b2 + f1 * d2 + f2,
    ]


def audit_pdf_capital_numeral_ink_heights(page):
    records = []
    unmapped = []
    font_cache = {}

    def visitor(text, cm, tm, font_dictionary, font_size):
        if not font_dictionary:
            return
        font_key = str(font_dictionary.get("/BaseFont"))
        if font_key not in font_cache:
            descendant = font_dictionary["/DescendantFonts"][0].get_object()
            descriptor = descendant["/FontDescriptor"].get_object()
            embedded_bytes = descriptor["/FontFile2"].get_object().get_data()
            font = TTFont(io.BytesIO(embedded_bytes))
            font_cache[font_key] = (font, font.getBestCmap(), font["head"].unitsPerEm)
        font, cmap, units_per_em = font_cache[font_key]
        combined = _multiply_pdf_matrices(tm, cm)
        vertical_scale = math.hypot(combined[2], combined[3])
        for character in text:
            if not (character.isupper() or character.isdigit()):
                continue
            glyph_name = cmap.get(ord(character))
            if glyph_name is None:
                unmapped.append((character, text, font_key))
                continue
            glyph = font["glyf"][glyph_name]
            ink_height_pt = (glyph.yMax - glyph.yMin) / units_per_em * font_size * vertical_scale
            records.append((ink_height_pt, character, text.strip(), font_size, font_key))

    page.extract_text(visitor_text=visitor)
    assert records and not unmapped, unmapped
    return records


pdf_glyph_ink_records = audit_pdf_capital_numeral_ink_heights(reader.pages[0])
minimum_pdf_glyph_ink_height_pt, minimum_glyph, minimum_glyph_text, minimum_glyph_tf, minimum_glyph_font = min(pdf_glyph_ink_records)
# The expanded 7.5 pt legend uses the 5 pt PDF glyph floor, not the legacy 2 mm ink-height recommendation.
assert all(r[3] >= 5.0 for r in pdf_glyph_ink_records)
with Image.open(export_paths["png"]) as image:
    width_px, height_px = image.size
    assert width_px >= 4000
    assert image.mode in {"RGB", "RGBA"}

qa_summary = {
    "pdf_pages": len(reader.pages),
    "png_pixels": (width_px, height_px),
    "datasets": int(data.loc[data.plot_flag, "dataset_id"].nunique()),
    "points": int(figure_data.loc[figure_data.plot_flag].shape[0]),
    "display_excluded_points": int(len(display_excluded)),
    "one_sided_limit_arrow_count": int(len(limit_arrow_annotations)),
    "filled_limit_caret_count": int(filled_limit_caret_count),
    "limit_arrow_linewidth_pt": float(LIMIT_ARROW_LINEWIDTH_PT),
    "boundary_arrow_count": int(len(boundary_arrow_annotations)),
    "errorbar_capsize_pt": float(PRIMARY_STYLES["bouwens2021_primary"]["capsize"]),
    "primary_errorbar_linewidth_pt": float(PRIMARY_STYLES["bouwens2021_primary"]["elinewidth"]),
    "comparison_errorbar_linewidth_pt": float(STYLE_BY_DATASET[comparison_dataset_ids[0]]["elinewidth"]),
    "minimum_pdf_glyph_ink_height_mm": minimum_pdf_glyph_ink_height_pt * 25.4 / 72,
    "minimum_central_marker_extent_mm": minimum_central_marker_extent_pt * 25.4 / 72,
    "primary_central_marker_extent_mm": primary_central_extent_pt * 25.4 / 72,
    "comparison_central_marker_extent_mm": comparison_central_extent_pt * 25.4 / 72,
    "theory_curve_count": int(len(fig.theory_line_artists)),
    "theory_grid_points_per_panel": int(THEORY_GRID_POINTS),
    "theory_model_labels": list(DISPLAY_MODEL_LABELS),
    "panel_slot_indices": list(PANEL_SLOT_INDICES),
    "legend_slot_index": int(LEGEND_SLOT_INDEX),
    "legend_vertical_gap_pt": float(fig.legend_vertical_gap_pt),
}
qa_summary

print("PASS: PDF glyph floor and all export files verified.")
display(fig)


## Final PRL guidance check

This is a readability alternative to the separate 2.04 mm submission-compliance version. Every primary and comparison central marker is intentionally 1.0 mm at final size. Comparison markers have transparent interiors so their uncertainty bars remain visible through the symbol. Error-bar capsize is 1.5 pt; primary and comparison error-bar line widths are 0.75 pt and 0.5 pt, respectively. In-range one-sided limits use unfilled 0.5 pt line arrows placed at zero horizontal offset, directly on the data/error-bar x coordinate; no filled Matplotlib caret remains. The arrow-direction vertical error-bar arm is not drawn for one-sided limits, preventing the two encodings from occupying the same segment; any reported opposite-direction arm is retained, and the source ledger remains unchanged. The Curtis-Lake Deep sample uses a neutral open hexagon so dataset identity is not confused with limit direction. One explicitly identified Harikane lower-limit row is omitted because its central anchor is below the fixed y-axis range; the source row remains in the TXT ledger. Chemerynska 2026 and Atek 2024 are hidden at the user request; rows remain in the ledger. No boundary-continuation arrows are drawn. Each panel displays two black theoretical UVLF curves: solid With PBH with the corrected 16-parameter MAP vector, and dotted No PBH using the corrected no-PBH astrophysical parameter vector. All are drawn beneath every observational artist after the exact inverse dust, bin-width, and AP transformations return them to the published observational frame. The legend occupies the former top-right panel, giving top-row z=6,7,8 plus legend and bottom-row z=9,10,11,12.5. The expanded literature legend is 7.5 pt; all PDF text runs retain the 5 pt floor. Because the 1.0 mm central markers are below the approximately 2 mm symbol-size recommendation used for the original PRL version, confirm this readability alternative with the journal before submission.
